In [2]:
from pathlib import Path

pasta_dados = Path("../dados/original")

arquivos_excel = list(pasta_dados.glob("*.xlsx"))
arquivos_excel += list(pasta_dados.glob("*.xls"))

for arquivo in arquivos_excel:
    print(arquivo.name)

base_original.xlsx


In [3]:
import pandas as pd

if not arquivos_excel:
    raise FileNotFoundError(
        "Nenhum arquivo Excel encontrado em dados/original."
    )

arquivo_base = arquivos_excel[0]

df = pd.read_excel(arquivo_base)

print("Arquivo carregado:", arquivo_base.name)
print("Base carregada com sucesso!")

Arquivo carregado: base_original.xlsx
Base carregada com sucesso!


In [4]:
df.head()

,Grupo de Serviço,Código do Fornecedor,Código do Serviço,Descrição do Item,Unidade medida
0,Civil,200982674,4033776,SV-EQUIPE TELHADO,SV
1,Civil,201464634,4012475,DIARIA RESGATISTA NIVEL 3 ATE 8H .,SV
2,Civil,200772986,4050934,SV OFICIAL MENSAL,SV
3,Civil,300623388,4097173,Pavimentação a frio por kg,KG
4,Civil,200982674,4066840,SV-EQUIPE PINTURA,SV


In [5]:
print("Quantidade de registros:", df.shape[0])
print("Quantidade de colunas:", df.shape[1])

Quantidade de registros: 9795
Quantidade de colunas: 5


In [6]:
print(df.columns.tolist())

['Grupo de Serviço', 'Código do Fornecedor', 'Código do Serviço', 'Descrição do Item', 'Unidade medida']


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9795 entries, 0 to 9794
Data columns (total 5 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   Grupo de Serviço      9795 non-null   str  
 1   Código do Fornecedor  9795 non-null   int64
 2   Código do Serviço     9795 non-null   int64
 3   Descrição do Item     9795 non-null   str  
 4   Unidade medida        9795 non-null   str  
dtypes: int64(2), str(3)
memory usage: 382.7 KB


In [8]:
df.isna().sum()

Grupo de Serviço        0
Código do Fornecedor    0
Código do Serviço       0
Descrição do Item       0
Unidade medida          0
dtype: int64

In [9]:

# Conferir os nomes e os valores de cada coluna
print("Colunas:")
for coluna in df.columns:
    print(f"- {coluna}")

print("\nPrimeiros registros:")
print(df.head(10).to_string(index=False))

print("\nTipos de dados:")
print(df.dtypes)

print("\nQuantidade de valores ausentes:")
print(df.isna().sum())

Colunas:
- Grupo de Serviço
- Código do Fornecedor
- Código do Serviço
- Descrição do Item
- Unidade medida

Primeiros registros:
Grupo de Serviço  Código do Fornecedor  Código do Serviço                        Descrição do Item Unidade medida
           Civil             200982674            4033776                        SV-EQUIPE TELHADO             SV
           Civil             201464634            4012475       DIARIA RESGATISTA NIVEL 3 ATE 8H .             SV
           Civil             200772986            4050934                        SV OFICIAL MENSAL             SV
           Civil             300623388            4097173               Pavimentação a frio por kg             KG
           Civil             200982674            4066840                        SV-EQUIPE PINTURA             SV
           Civil             200772986            4050928                    SV ENCARREGADO MENSAL             SV
           Civil             201332350            4096183               

In [10]:

componentes = [
    "Grupo de Serviço",
    "Código do Fornecedor",
    "Código do Serviço",
    "Unidade medida"
]

print("Registros duplicados considerando os quatro componentes:")
print(df.duplicated(subset=componentes, keep=False).sum())

print("\nDescrições de item repetidas:")
print(df["Descrição do Item"].duplicated(keep=False).sum())

Registros duplicados considerando os quatro componentes:
0

Descrições de item repetidas:
3765


In [11]:

# Quantidade total de registros e descrições distintas
print("Total de registros:", len(df))
print(
    "Descrições distintas:",
    df["Descrição do Item"].nunique()
)

# Descrições que aparecem mais de uma vez
contagem_descricoes = (
    df["Descrição do Item"]
    .value_counts()
)

repetidas = contagem_descricoes[
    contagem_descricoes > 1
]

print("Descrições que se repetem:", len(repetidas))
print("Registros pertencentes a descrições repetidas:",
      repetidas.sum())

# Exibir as 20 descrições mais frequentes
print("\n20 descrições mais frequentes:")
print(repetidas.head(20).to_string())

Total de registros: 9795
Descrições distintas: 7468
Descrições que se repetem: 1438
Registros pertencentes a descrições repetidas: 3765

20 descrições mais frequentes:
Descrição do Item
SV - FURAÇÃO EM CONCRETO COM COROA DIAM     13
SV - ANCORAGEM QUÍMICA SISTEMA HVA DA HI    13
PREMIO ASSIDUIDADE                           8
MECÂNICO DE MANUTENÇÃO (HORA NORMAL)         7
SOLDADOR (HORA NORMAL)                       7
TÉCNICO DE SEGURANÇA (HORA NORMAL)           7
TÉCNICO DE SEGURANÇA (HE DOM/FER)            7
TÉCNICO DE SEGURANÇA (HORA EXTRA SEM)        7
MECÂNICO DE MANUTENÇÃO (HORA EXTRA SEM)      7
MECÂNICO DE MANUT. (HORA EXTRA DOM/FER)      7
MECÂNICO DE MANUTENÇÃO (INSALUBRIDADE)       7
MECÂNICO DE MANUT. (PRERICULOSIDADE)         7
ELETRICISTA DE MANUTENÇÃO (HORA NORMAL)      7
ELETRICISTA DE MANUT. (HORA EXTRA SEM)       7
ELETRICISTA DE MANUT. (HE DOM/FER)           7
SV - ANCORAGEM QUÍMICA SISTEMA HIT - FOR     6
MONTADOR (HORA NORMAL)                       6
ENCARREGADO DE 

In [16]:

from rapidfuzz import fuzz, process

# Usar apenas descrições únicas para evitar comparações repetidas
descricoes = (
    df["Descrição do Item"]
    .dropna()
    .drop_duplicates()
    .tolist()
)

resultados = []

for descricao in descricoes:
    candidatos = process.extract(
        descricao,
        descricoes,
        scorer=fuzz.ratio,
        limit=5,
        score_cutoff=75
    )

    for candidato, pontuacao, _ in candidatos:
        # Evitar comparar a descrição com ela mesma
        if descricao != candidato:
            resultados.append({
                "Descricao original": descricao,
                "Descricao candidata": candidato,
                "Similaridade": round(pontuacao, 2)
            })

pares = pd.DataFrame(resultados)

if not pares.empty:
    pares = pares.drop_duplicates(
        subset=["Descricao original", "Descricao candidata"]
    )
    pares = pares[
        pares["Descricao original"] < pares["Descricao candidata"]
    ]
    pares = pares.sort_values(
        "Similaridade", ascending=False
    )

display(pares.head(30))

,Descricao original,Descricao candidata,Similaridade
9069,"Telha galvinizada TP40 com espess.0,5mm","Telha galvinizada TP40 com espess.0,65mm",98.73
17501,H.I+E.DF-Of. de Manut. Predial NR35-ADM,H.I+E.DF-Of. de Manut. Predial NR35-ADM,98.73
16636,H.P.E.DF-Of. de Manut. Predial NR35-ADM,H.P.E.DF-Of. de Manut. Predial NR35-ADM,98.73
554,SV - ANCORAGEM QUÍMICA SISTEMA HVA DA H,SV - ANCORAGEM QUÍMICA SISTEMA HVA DA HI,98.73
15667,SV-LIMP E CONSERV. Aux de Serviç HE 100%,SV-LIMP E CONSERV.Aux de Serviç HE 100%,98.73
17057,H.I+E.SS-Of. de Manut. Predial NR35-ADM,H.I+E.SS-Of. de Manut. Predial NR35-ADM,98.73
4907,LOCAÇÃO DE MISTURADOR PLANETÁRIO HORIZO,LOCAÇÃO DE MISTURADOR PLANETÁRIO HORIZON,98.73
316,SV - ANCORAGEM QUÍMICA SISTEMA HIT - FOR,SV - ANCORAGEM QUÍMICA SISTEMA HIT- FOR,98.73
16397,H.P.E.SS-Of. de Manut. Predial NR35-ADM,H.P.E.SS-Of. de Manut. Predial NR35-ADM,98.73
14209,"""SV - FORRO PVC MODULAR 625 x 625MM OU 6",SV - FORRO PVC MODULAR 625 x 625MM OU 6,98.73


In [17]:

import re

def extrair_numeros(texto):
    # Captura números inteiros e decimais, inclusive medidas
    return re.findall(r"\d+(?:[.,]\d+)?", str(texto))

pares["Numeros original"] = pares["Descricao original"].apply(
    extrair_numeros
)
pares["Numeros candidato"] = pares["Descricao candidata"].apply(
    extrair_numeros
)

pares["Possui diferenca numerica"] = pares.apply(
    lambda linha: (
        linha["Numeros original"] != linha["Numeros candidato"]
    ),
    axis=1
)

print("Pares com diferenças numéricas:")
display(
    pares[pares["Possui diferenca numerica"]][
        [
            "Descricao original",
            "Descricao candidata",
            "Similaridade",
            "Numeros original",
            "Numeros candidato"
        ]
    ].head(30)
)

Pares com diferenças numéricas:


,Descricao original,Descricao candidata,Similaridade,Numeros original,Numeros candidato
9069,"Telha galvinizada TP40 com espess.0,5mm","Telha galvinizada TP40 com espess.0,65mm",98.73,"[40, 0,5]","[40, 0,65]"
19433,Inst. e Ativ. de cam. IP Dome - 4 cam.,Inst. e Ativ. de cam. IP Dome - 64 cam.,98.70,[4],[64]
19425,Inst. e Ativ. de cam. IP Dome - 1 cam.,Inst. e Ativ. de cam. IP Dome - 16 cam.,98.70,[1],[16]
16834,SV - Container p/ Combustível 50L - CSG,SV - Container p/ Combustível 5L - CSG,98.70,[50],[5]
2892,SV TUBO RÍGIDO COM PONTA E BOLSA 150 MM,SV TUBO RÍGIDO COM PONTA E BOLSA 50 MM,98.70,[150],[50]
19429,Inst. e Ativ. de cam. IP Dome - 2 cam.,Inst. e Ativ. de cam. IP Dome - 32 cam.,98.70,[2],[32]
11709,SV TRECHO TRILHO 12M s/linha de vida,SV TRECHO TRILHO 132M s/linha de vida,98.63,[12],[132]
9726,SV TRECHO TRILHO 12M c/linha de vida,SV TRECHO TRILHO 132M c/linha de vida,98.63,[12],[132]
9742,SV TRECHO TRILHO 60M c/linha de vida,SV TRECHO TRILHO 6M c/linha de vida,98.59,[60],[6]
9734,SV TRECHO TRILHO 36M c/linha de vida,SV TRECHO TRILHO 6M c/linha de vida,98.59,[36],[6]


In [18]:

import re
import pandas as pd

def extrair_numeros(texto):
    # Preserva números inteiros e decimais, inclusive com vírgula
    return re.findall(r"\d+(?:[.,]\d+)?", str(texto))

pares["Numeros original"] = pares["Descricao original"].apply(
    extrair_numeros
)
pares["Numeros candidato"] = pares["Descricao candidata"].apply(
    extrair_numeros
)

pares["Classificacao"] = pares.apply(
    lambda linha: (
        "Revisar: números diferentes"
        if linha["Numeros original"] != linha["Numeros candidato"]
        else "Analisar: números iguais"
    ),
    axis=1
)

print("Quantidade por classificação:")
print(pares["Classificacao"].value_counts())

print("\nPares com números iguais:")
display(
    pares[pares["Classificacao"] == "Analisar: números iguais"][
        [
            "Descricao original",
            "Descricao candidata",
            "Similaridade"
        ]
    ].head(20)
)

print("\nPares com números diferentes:")
display(
    pares[pares["Classificacao"] == "Revisar: números diferentes"][
        [
            "Descricao original",
            "Descricao candidata",
            "Similaridade",
            "Numeros original",
            "Numeros candidato"
        ]
    ].head(20)
)

Quantidade por classificação:
Classificacao
Analisar: números iguais       6140
Revisar: números diferentes    3638
Name: count, dtype: int64

Pares com números iguais:


,Descricao original,Descricao candidata,Similaridade
17501,H.I+E.DF-Of. de Manut. Predial NR35-ADM,H.I+E.DF-Of. de Manut. Predial NR35-ADM,98.73
16636,H.P.E.DF-Of. de Manut. Predial NR35-ADM,H.P.E.DF-Of. de Manut. Predial NR35-ADM,98.73
554,SV - ANCORAGEM QUÍMICA SISTEMA HVA DA H,SV - ANCORAGEM QUÍMICA SISTEMA HVA DA HI,98.73
15667,SV-LIMP E CONSERV. Aux de Serviç HE 100%,SV-LIMP E CONSERV.Aux de Serviç HE 100%,98.73
17057,H.I+E.SS-Of. de Manut. Predial NR35-ADM,H.I+E.SS-Of. de Manut. Predial NR35-ADM,98.73
4907,LOCAÇÃO DE MISTURADOR PLANETÁRIO HORIZO,LOCAÇÃO DE MISTURADOR PLANETÁRIO HORIZON,98.73
316,SV - ANCORAGEM QUÍMICA SISTEMA HIT - FOR,SV - ANCORAGEM QUÍMICA SISTEMA HIT- FOR,98.73
16397,H.P.E.SS-Of. de Manut. Predial NR35-ADM,H.P.E.SS-Of. de Manut. Predial NR35-ADM,98.73
14209,"""SV - FORRO PVC MODULAR 625 x 625MM OU 6",SV - FORRO PVC MODULAR 625 x 625MM OU 6,98.73
7028,SV-PINTURA TINTA ESMALTE BASE ÁGUA - ES,SV-PINTURA TINTA ESMALTE BASE ÁGUA - ESQ,98.73



Pares com números diferentes:


,Descricao original,Descricao candidata,Similaridade,Numeros original,Numeros candidato
9069,"Telha galvinizada TP40 com espess.0,5mm","Telha galvinizada TP40 com espess.0,65mm",98.73,"[40, 0,5]","[40, 0,65]"
19433,Inst. e Ativ. de cam. IP Dome - 4 cam.,Inst. e Ativ. de cam. IP Dome - 64 cam.,98.70,[4],[64]
19425,Inst. e Ativ. de cam. IP Dome - 1 cam.,Inst. e Ativ. de cam. IP Dome - 16 cam.,98.70,[1],[16]
16834,SV - Container p/ Combustível 50L - CSG,SV - Container p/ Combustível 5L - CSG,98.70,[50],[5]
2892,SV TUBO RÍGIDO COM PONTA E BOLSA 150 MM,SV TUBO RÍGIDO COM PONTA E BOLSA 50 MM,98.70,[150],[50]
19429,Inst. e Ativ. de cam. IP Dome - 2 cam.,Inst. e Ativ. de cam. IP Dome - 32 cam.,98.70,[2],[32]
11709,SV TRECHO TRILHO 12M s/linha de vida,SV TRECHO TRILHO 132M s/linha de vida,98.63,[12],[132]
9726,SV TRECHO TRILHO 12M c/linha de vida,SV TRECHO TRILHO 132M c/linha de vida,98.63,[12],[132]
9742,SV TRECHO TRILHO 60M c/linha de vida,SV TRECHO TRILHO 6M c/linha de vida,98.59,[60],[6]
9734,SV TRECHO TRILHO 36M c/linha de vida,SV TRECHO TRILHO 6M c/linha de vida,98.59,[36],[6]


In [19]:

# Identificar pares que ficam iguais após a normalização conservadora
pares["Normalizada original"] = (
    pares["Descricao original"].apply(normalizar_descricao)
)
pares["Normalizada candidata"] = (
    pares["Descricao candidata"].apply(normalizar_descricao)
)

pares["Iguais apos normalizacao"] = (
    pares["Normalizada original"]
    == pares["Normalizada candidata"]
)

print("Total de pares:", len(pares))
print(
    "Pares iguais após normalização:",
    pares["Iguais apos normalizacao"].sum()
)
print(
    "Pares ainda diferentes após normalização:",
    (~pares["Iguais apos normalizacao"]).sum()
)

print("\nExemplos que ficaram iguais após normalização:")
display(
    pares[pares["Iguais apos normalizacao"]][
        [
            "Descricao original",
            "Descricao candidata",
            "Similaridade"
        ]
    ].head(15)
)

Total de pares: 9778
Pares iguais após normalização: 36
Pares ainda diferentes após normalização: 9742

Exemplos que ficaram iguais após normalização:


,Descricao original,Descricao candidata,Similaridade
17501,H.I+E.DF-Of. de Manut. Predial NR35-ADM,H.I+E.DF-Of. de Manut. Predial NR35-ADM,98.73
16636,H.P.E.DF-Of. de Manut. Predial NR35-ADM,H.P.E.DF-Of. de Manut. Predial NR35-ADM,98.73
17057,H.I+E.SS-Of. de Manut. Predial NR35-ADM,H.I+E.SS-Of. de Manut. Predial NR35-ADM,98.73
16397,H.P.E.SS-Of. de Manut. Predial NR35-ADM,H.P.E.SS-Of. de Manut. Predial NR35-ADM,98.73
16157,H.E.DF-Of. de Manut. Predial NR35-ADM,H.E.DF-Of. de Manut. Predial NR35-ADM,98.67
16389,Peric.Mês - Aux. de Sv. Gerais - ADM,Peric.Mês - Aux. de Sv. Gerais - ADM,98.67
15785,H.E.SS-Of. de Manut. Predial NR35-ADM,H.E.SS-Of. de Manut. Predial NR35-ADM,98.67
15106,HN+Add.N - Aux. de Sv. Gerais - ADM,HN+Add.N - Aux. de Sv. Gerais - ADM,98.63
16628,Peric.Mês - Aux. de Sv. Gerais - XY,Peric.Mês - Aux. de Sv. Gerais - XY,98.59
15197,Peric.Mês - Op. de motosserra - ADM,Peric.Mês - Op. de motosserra - ADM,98.59


In [20]:

# Agrupar descrições originais pela versão normalizada
grupos_normalizados = (
    df.groupby("Descricao_Normalizada")
    .agg(
        Quantidade_registros=("Descrição do Item", "size"),
        Quantidade_descricoes_originais=("Descrição do Item", "nunique"),
        Descricoes_originais=(
            "Descrição do Item",
            lambda coluna: sorted(coluna.unique())
        )
    )
    .reset_index()
)

# Manter grupos com mais de uma descrição original
possiveis_duplicidades_formatacao = grupos_normalizados[
    grupos_normalizados["Quantidade_descricoes_originais"] > 1
].copy()

# Ordenar pelos grupos com mais descrições originais
possiveis_duplicidades_formatacao = (
    possiveis_duplicidades_formatacao
    .sort_values(
        by=["Quantidade_descricoes_originais", "Quantidade_registros"],
        ascending=False
    )
)

print(
    "Grupos com descrições equivalentes após normalização:",
    len(possiveis_duplicidades_formatacao)
)

display(possiveis_duplicidades_formatacao.head(20))

Grupos com descrições equivalentes após normalização: 37


,Descricao_Normalizada,Quantidade_registros,Quantidade_descricoes_originais,Descricoes_originais
7138,tecnico de seguranca (hora normal),13,2,"[TECNICO DE SEGURANCA (HORA NORMAL), TÉCNICO D..."
7139,tecnico de seguranca (hora not),12,2,"[TECNICO DE SEGURANCA (HORA NOT), TÉCNICO DE S..."
230,artifice,4,2,"[ARTIFICE, ARTÍFICE]"
3159,peric.mes - aux. de sv. gerais - adm,4,2,"[Peric.Mês - Aux. de Sv. Gerais - ADM, Peric..."
7133,tecnico de seguranca,4,2,"[Tecnico de segurança, Técnico de Segurança]"
700,carpinteiro - hora normal a disposicao,3,2,"[CARPINTEIRO - HORA NORMAL A DISPOSICAO, CARPI..."
2344,lider,3,2,"[LIDER, Líder]"
2654,mes-of. de manut. predial nr35-adm,3,2,"[Mês-Of. de Manut. Predial NR35-ADM, Mês-Of. ..."
3161,peric.mes - aux. de sv. gerais - xy,3,2,"[Peric.Mês - Aux. de Sv. Gerais - XY, Peric.M..."
5939,sv-adaptador soldavel com flange p/cx. d,3,2,"[SV-ADAPTADOR SOLDAVEL COM FLANGE P/CX. D, SV-..."


In [21]:

# Identificar as descrições normalizadas que possuem
# mais de uma versão original
normalizadas_repetidas = set(
    possiveis_duplicidades_formatacao["Descricao_Normalizada"]
)

# Recuperar todos os registros correspondentes na base original
registros_para_revisar = df[
    df["Descricao_Normalizada"].isin(normalizadas_repetidas)
].copy()

# Selecionar as colunas relevantes para a análise
colunas_analise = [
    "Grupo de Serviço",
    "Código do Fornecedor",
    "Código do Serviço",
    "Descrição do Item",
    "Unidade medida",
    "Descricao_Normalizada"
]

registros_para_revisar = registros_para_revisar[
    colunas_analise
].sort_values(
    by=["Descricao_Normalizada", "Código do Serviço"]
)

print("Total de registros para revisão:", len(registros_para_revisar))

display(registros_para_revisar.head(50))

Total de registros para revisão: 108


,Grupo de Serviço,Código do Fornecedor,Código do Serviço,Descrição do Item,Unidade medida,Descricao_Normalizada
2469,Civil,200772986,4086466,ARMADOR - HORA NORMAL A DISPOSICAO,H,armador - hora normal a disposicao
3374,Civil,201332350,4096457,ARMADOR - HORA NORMAL À DISPOSIÇÃO,H,armador - hora normal a disposicao
8721,Facilities,200880612,4092270,ARTÍFICE,SV,artifice
7871,Facilities,200364916,4099911,ARTIFICE,SV,artifice
7943,Facilities,200733106,4100964,ARTÍFICE,SV,artifice
8770,Facilities,201418204,4117541,ARTIFICE,SV,artifice
2455,Civil,200772986,4086449,CARPINTEIRO - HORA NORMAL A DISPOSICAO,H,carpinteiro - hora normal a disposicao
3364,Civil,201332350,4096445,CARPINTEIRO - HORA NORMAL À DISPOSIÇÃO,H,carpinteiro - hora normal a disposicao
3568,Civil,300343622,4100183,CARPINTEIRO - HORA NORMAL À DISPOSIÇÃO,H,carpinteiro - hora normal a disposicao
9742,Facilities,201879426,4068601,COMBUSTIVEL,SV,combustivel


In [22]:

# Resumir os grupos de descrições equivalentes
resumo_grupos = (
    registros_para_revisar
    .groupby("Descricao_Normalizada")
    .agg(
        Quantidade_registros=("Código do Serviço", "size"),
        Quantidade_codigos_servico=("Código do Serviço", "nunique"),
        Quantidade_fornecedores=("Código do Fornecedor", "nunique"),
        Quantidade_grupos_servico=("Grupo de Serviço", "nunique"),
        Quantidade_unidades=("Unidade medida", "nunique"),
        Descricoes_originais=(
            "Descrição do Item",
            lambda coluna: list(coluna.unique())
        )
    )
    .reset_index()
)

# Criar uma classificação inicial para orientar a revisão
def classificar_grupo(linha):
    if linha["Quantidade_codigos_servico"] == 1:
        return "Mesmo código de serviço"
    return "Múltiplos códigos de serviço"

resumo_grupos["Classificacao_inicial"] = (
    resumo_grupos.apply(classificar_grupo, axis=1)
)

resumo_grupos = resumo_grupos.sort_values(
    by=[
        "Quantidade_codigos_servico",
        "Quantidade_registros"
    ],
    ascending=False
)

print("Total de grupos:", len(resumo_grupos))

print("\nResumo da classificação:")
display(
    resumo_grupos["Classificacao_inicial"]
    .value_counts()
    .rename_axis("Classificação")
    .reset_index(name="Quantidade de grupos")
)

print("\nGrupos com múltiplos códigos de serviço:")
display(
    resumo_grupos[
        resumo_grupos["Quantidade_codigos_servico"] > 1
    ].head(20)
)

Total de grupos: 37

Resumo da classificação:


,Classificação,Quantidade de grupos
0,Múltiplos códigos de serviço,37



Grupos com múltiplos códigos de serviço:


,Descricao_Normalizada,Quantidade_registros,Quantidade_codigos_servico,Quantidade_fornecedores,Quantidade_grupos_servico,Quantidade_unidades,Descricoes_originais,Classificacao_inicial
34,tecnico de seguranca (hora normal),13,4,10,1,1,"[TECNICO DE SEGURANCA (HORA NORMAL), TÉCNICO D...",Múltiplos códigos de serviço
35,tecnico de seguranca (hora not),12,4,9,1,1,"[TECNICO DE SEGURANCA (HORA NOT), TÉCNICO DE S...",Múltiplos códigos de serviço
1,artifice,4,4,4,1,1,"[ARTÍFICE, ARTIFICE]",Múltiplos códigos de serviço
27,peric.mes - aux. de sv. gerais - adm,4,4,1,1,1,"[Peric.Mês - Aux. de Sv. Gerais - ADM, Peric....",Múltiplos códigos de serviço
33,tecnico de seguranca,4,3,4,3,1,"[Técnico de Segurança, Tecnico de segurança]",Múltiplos códigos de serviço
2,carpinteiro - hora normal a disposicao,3,3,3,1,1,"[CARPINTEIRO - HORA NORMAL A DISPOSICAO, CARPI...",Múltiplos códigos de serviço
26,mes-of. de manut. predial nr35-adm,3,3,1,1,1,"[Mês-Of. de Manut. Predial NR35-ADM, Mês-Of. d...",Múltiplos códigos de serviço
28,peric.mes - aux. de sv. gerais - xy,3,3,1,1,1,"[Peric.Mês - Aux. de Sv. Gerais - XY, Peric.Mê...",Múltiplos códigos de serviço
31,sv-adaptador soldavel com flange p/cx. d,3,3,1,1,1,"[SV-ADAPTADOR SOLDÁVEL COM FLANGE P/CX. D, SV-...",Múltiplos códigos de serviço
32,sv-demolicao de concreto armado com mart,3,3,1,1,1,"[SV-DEMOLIÇÃO DE CONCRETO ARMADO COM MART, SV-...",Múltiplos códigos de serviço


In [23]:

# Investigar possíveis duplicidades dentro do mesmo fornecedor
investigacao_fornecedor = (
    registros_para_revisar
    .groupby([
        "Descricao_Normalizada",
        "Código do Fornecedor"
    ])
    .agg(
        Quantidade_registros=("Código do Serviço", "size"),
        Quantidade_codigos_servico=("Código do Serviço", "nunique"),
        Codigos_servico=(
            "Código do Serviço",
            lambda coluna: sorted(coluna.unique())
        ),
        Descricoes_originais=(
            "Descrição do Item",
            lambda coluna: list(coluna.unique())
        ),
        Grupos_servico=(
            "Grupo de Serviço",
            lambda coluna: list(coluna.unique())
        ),
        Unidades=(
            "Unidade medida",
            lambda coluna: list(coluna.unique())
        )
    )
    .reset_index()
)

# Manter grupos do mesmo fornecedor com códigos de serviço diferentes
possiveis_duplicidades_mesmo_fornecedor = (
    investigacao_fornecedor[
        investigacao_fornecedor["Quantidade_codigos_servico"] > 1
    ]
    .sort_values(
        by="Quantidade_codigos_servico",
        ascending=False
    )
)

print(
    "Grupos do mesmo fornecedor com múltiplos códigos:",
    len(possiveis_duplicidades_mesmo_fornecedor)
)

display(possiveis_duplicidades_mesmo_fornecedor)

Grupos do mesmo fornecedor com múltiplos códigos: 31


,Descricao_Normalizada,Código do Fornecedor,Quantidade_registros,Quantidade_codigos_servico,Codigos_servico,Descricoes_originais,Grupos_servico,Unidades
55,tecnico de seguranca (hora normal),300745236,4,4,"[4076416, 4083623, 4089078, 4089824]","[TECNICO DE SEGURANCA (HORA NORMAL), TÉCNICO D...",[Eletromecânica],[H]
64,tecnico de seguranca (hora not),300745236,4,4,"[4076417, 4083624, 4089079, 4089825]","[TECNICO DE SEGURANCA (HORA NOT), TÉCNICO DE S...",[Eletromecânica],[H]
36,peric.mes - aux. de sv. gerais - adm,300380700,4,4,"[4085954, 4085991, 4086228, 4086285]","[Peric.Mês - Aux. de Sv. Gerais - ADM, Peric....",[Facilities],[SV]
35,mes-of. de manut. predial nr35-adm,300380700,3,3,"[4085737, 4085742, 4085755]","[Mês-Of. de Manut. Predial NR35-ADM, Mês-Of. d...",[Facilities],[SV]
65,veiculo,200323568,3,3,"[4099981, 4099993, 4099998]","[VEÍCULO, VEICULO]",[Facilities],[SV]
37,peric.mes - aux. de sv. gerais - xy,300380700,3,3,"[4085753, 4086051, 4086322]","[Peric.Mês - Aux. de Sv. Gerais - XY, Peric.Mê...",[Facilities],[SV]
40,sv-adaptador soldavel com flange p/cx. d,300056198,3,3,"[4023817, 4023818, 4023821]","[SV-ADAPTADOR SOLDÁVEL COM FLANGE P/CX. D, SV-...",[Civil],[ST]
41,sv-demolicao de concreto armado com mart,300056198,3,3,"[4023880, 4023881, 4023882]","[SV-DEMOLIÇÃO DE CONCRETO ARMADO COM MART, SV-...",[Civil],[M3]
11,h.e.df-aux. de sv. gerais -adm,300380700,2,2,"[4085908, 4085909]","[H.E.DF-Aux. de Sv. Gerais -ADM, H.E.DF-Aux. ...",[Facilities],[SV]
12,h.e.df-of. de manut. predial nr35-adm,300380700,2,2,"[4085917, 4085935]","[H.E.DF-Of. de Manut. Predial NR35-ADM, H.E.DF...",[Facilities],[SV]


In [24]:

# Identificar os grupos que possuem mais de um código
# de serviço para a mesma descrição normalizada e fornecedor
chaves_suspeitas = possiveis_duplicidades_mesmo_fornecedor[
    ["Descricao_Normalizada", "Código do Fornecedor"]
].drop_duplicates()

# Recuperar os registros completos desses grupos
registros_suspeitos = registros_para_revisar.merge(
    chaves_suspeitas,
    on=["Descricao_Normalizada", "Código do Fornecedor"],
    how="inner"
)

print("Grupos suspeitos:", len(chaves_suspeitas))
print("Registros envolvidos:", len(registros_suspeitos))
print(
    "Códigos de serviço distintos:",
    registros_suspeitos["Código do Serviço"].nunique()
)

print(
    "\nPercentual da base envolvido:",
    round(len(registros_suspeitos) / len(df) * 100, 2),
    "%"
)

# Exibir os registros para conferência
display(
    registros_suspeitos[
        [
            "Grupo de Serviço",
            "Código do Fornecedor",
            "Código do Serviço",
            "Descrição do Item",
            "Unidade medida"
        ]
    ].sort_values(
        by=["Código do Fornecedor", "Descrição do Item"]
    )
)

Grupos suspeitos: 31
Registros envolvidos: 73
Códigos de serviço distintos: 73

Percentual da base envolvido: 0.75 %


,Grupo de Serviço,Código do Fornecedor,Código do Serviço,Descrição do Item,Unidade medida
71,Facilities,200323568,4099993,VEICULO,SV
72,Facilities,200323568,4099998,VEICULO,SV
70,Facilities,200323568,4099981,VEÍCULO,SV
54,Civil,300056198,4023819,SV-ADAPTADOR SOLDAVEL COM FLANGE P/CX.,ST
57,Civil,300056198,4023818,SV-ADAPTADOR SOLDAVEL COM FLANGE P/CX. D,ST
...,...,...,...,...,...
68,Eletromecânica,300745236,4089079,TECNICO DE SEGURANCA (HORA NOT),H
63,Eletromecânica,300745236,4083623,TÉCNICO DE SEGURANÇA (HORA NORMAL),H
65,Eletromecânica,300745236,4089824,TÉCNICO DE SEGURANÇA (HORA NORMAL),H
67,Eletromecânica,300745236,4083624,TÉCNICO DE SEGURANÇA (HORA NOT),H


In [25]:

# Separar pares que continuam diferentes após a normalização
# e possuem similaridade textual elevada
pares_para_revisar = pares[
    (~pares["Iguais apos normalizacao"])
    & (pares["Similaridade"] >= 90)
].copy()

# Colocar lado a lado os textos e os números extraídos
colunas = [
    "Descricao original",
    "Descricao candidata",
    "Similaridade",
    "Numeros original",
    "Numeros candidato",
    "Possui diferenca numerica"
]

pares_para_revisar = pares_para_revisar.sort_values(
    by="Similaridade",
    ascending=False
)

print(
    "Pares diferentes após normalização com similaridade >= 90:",
    len(pares_para_revisar)
)

display(pares_para_revisar[colunas].head(30))

Pares diferentes após normalização com similaridade >= 90: 4048


,Descricao original,Descricao candidata,Similaridade,Numeros original,Numeros candidato,Possui diferenca numerica
9069,"Telha galvinizada TP40 com espess.0,5mm","Telha galvinizada TP40 com espess.0,65mm",98.73,"[40, 0,5]","[40, 0,65]",True
554,SV - ANCORAGEM QUÍMICA SISTEMA HVA DA H,SV - ANCORAGEM QUÍMICA SISTEMA HVA DA HI,98.73,[],[],False
15667,SV-LIMP E CONSERV. Aux de Serviç HE 100%,SV-LIMP E CONSERV.Aux de Serviç HE 100%,98.73,[100],[100],False
4907,LOCAÇÃO DE MISTURADOR PLANETÁRIO HORIZO,LOCAÇÃO DE MISTURADOR PLANETÁRIO HORIZON,98.73,[],[],False
316,SV - ANCORAGEM QUÍMICA SISTEMA HIT - FOR,SV - ANCORAGEM QUÍMICA SISTEMA HIT- FOR,98.73,[],[],False
14209,"""SV - FORRO PVC MODULAR 625 x 625MM OU 6",SV - FORRO PVC MODULAR 625 x 625MM OU 6,98.73,"[625, 625, 6]","[625, 625, 6]",False
7028,SV-PINTURA TINTA ESMALTE BASE ÁGUA - ES,SV-PINTURA TINTA ESMALTE BASE ÁGUA - ESQ,98.73,[],[],False
165,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,SV - FURAÇÃO EM CONCRETO COM COROA DIAMA,98.73,[],[],False
19433,Inst. e Ativ. de cam. IP Dome - 4 cam.,Inst. e Ativ. de cam. IP Dome - 64 cam.,98.70,[4],[64],True
19425,Inst. e Ativ. de cam. IP Dome - 1 cam.,Inst. e Ativ. de cam. IP Dome - 16 cam.,98.70,[1],[16],True


In [26]:

# Resumo dos pares com similaridade igual ou superior a 90%

print("Total de pares para análise:", len(pares_para_revisar))

print("\nDiferença nos números:")
display(
    pares_para_revisar["Possui diferenca numerica"]
    .value_counts()
    .rename(index={
        True: "Possuem números diferentes",
        False: "Possuem os mesmos números"
    })
    .to_frame("Quantidade")
)

print("\nExemplos com números diferentes:")
display(
    pares_para_revisar[
        pares_para_revisar["Possui diferenca numerica"]
    ][
        [
            "Descricao original",
            "Descricao candidata",
            "Similaridade",
            "Numeros original",
            "Numeros candidato"
        ]
    ].head(15)
)

print("\nExemplos sem diferenças numéricas:")
display(
    pares_para_revisar[
        ~pares_para_revisar["Possui diferenca numerica"]
    ][
        [
            "Descricao original",
            "Descricao candidata",
            "Similaridade"
        ]
    ].head(15)
)

Total de pares para análise: 4048

Diferença nos números:


,Quantidade
Possui diferenca numerica,
Possuem números diferentes,2281
Possuem os mesmos números,1767



Exemplos com números diferentes:


,Descricao original,Descricao candidata,Similaridade,Numeros original,Numeros candidato
9069,"Telha galvinizada TP40 com espess.0,5mm","Telha galvinizada TP40 com espess.0,65mm",98.73,"[40, 0,5]","[40, 0,65]"
19433,Inst. e Ativ. de cam. IP Dome - 4 cam.,Inst. e Ativ. de cam. IP Dome - 64 cam.,98.70,[4],[64]
19425,Inst. e Ativ. de cam. IP Dome - 1 cam.,Inst. e Ativ. de cam. IP Dome - 16 cam.,98.70,[1],[16]
16834,SV - Container p/ Combustível 50L - CSG,SV - Container p/ Combustível 5L - CSG,98.70,[50],[5]
2892,SV TUBO RÍGIDO COM PONTA E BOLSA 150 MM,SV TUBO RÍGIDO COM PONTA E BOLSA 50 MM,98.70,[150],[50]
19429,Inst. e Ativ. de cam. IP Dome - 2 cam.,Inst. e Ativ. de cam. IP Dome - 32 cam.,98.70,[2],[32]
11709,SV TRECHO TRILHO 12M s/linha de vida,SV TRECHO TRILHO 132M s/linha de vida,98.63,[12],[132]
9726,SV TRECHO TRILHO 12M c/linha de vida,SV TRECHO TRILHO 132M c/linha de vida,98.63,[12],[132]
9742,SV TRECHO TRILHO 60M c/linha de vida,SV TRECHO TRILHO 6M c/linha de vida,98.59,[60],[6]
9734,SV TRECHO TRILHO 36M c/linha de vida,SV TRECHO TRILHO 6M c/linha de vida,98.59,[36],[6]



Exemplos sem diferenças numéricas:


,Descricao original,Descricao candidata,Similaridade
554,SV - ANCORAGEM QUÍMICA SISTEMA HVA DA H,SV - ANCORAGEM QUÍMICA SISTEMA HVA DA HI,98.73
15667,SV-LIMP E CONSERV. Aux de Serviç HE 100%,SV-LIMP E CONSERV.Aux de Serviç HE 100%,98.73
4907,LOCAÇÃO DE MISTURADOR PLANETÁRIO HORIZO,LOCAÇÃO DE MISTURADOR PLANETÁRIO HORIZON,98.73
316,SV - ANCORAGEM QUÍMICA SISTEMA HIT - FOR,SV - ANCORAGEM QUÍMICA SISTEMA HIT- FOR,98.73
14209,"""SV - FORRO PVC MODULAR 625 x 625MM OU 6",SV - FORRO PVC MODULAR 625 x 625MM OU 6,98.73
7028,SV-PINTURA TINTA ESMALTE BASE ÁGUA - ES,SV-PINTURA TINTA ESMALTE BASE ÁGUA - ESQ,98.73
165,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,SV - FURAÇÃO EM CONCRETO COM COROA DIAMA,98.73
16193,SV-MAQUINA LAVADORA A5 TREFILA I,SV-MAQUINA LAVADORA A5 TREFILA II,98.46
16870,H.P+Ad.Not-Op. de motosserra -ADM,H.P+Ad.Not-Op. de motosserra-ADM,98.46
17672,H.I+A.Not-Op. de motosserra -ADM,H.I+A.Not-Op. de motosserra-ADM,98.41


In [27]:

# Recuperar os dados dos serviços relacionados aos pares candidatos

colunas_servico = [
    "Grupo de Serviço",
    "Código do Fornecedor",
    "Código do Serviço",
    "Descrição do Item",
    "Unidade medida"
]

# Uma linha por combinação de descrição e componentes
servicos = df[colunas_servico].drop_duplicates().copy()

# Dados associados à descrição original
originais = servicos.rename(columns={
    "Grupo de Serviço": "Grupo original",
    "Código do Fornecedor": "Fornecedor original",
    "Código do Serviço": "Código original",
    "Descrição do Item": "Descricao original",
    "Unidade medida": "Unidade original"
})

# Dados associados à descrição candidata
candidatos = servicos.rename(columns={
    "Grupo de Serviço": "Grupo candidato",
    "Código do Fornecedor": "Fornecedor candidato",
    "Código do Serviço": "Código candidato",
    "Descrição do Item": "Descricao candidata",
    "Unidade medida": "Unidade candidata"
})

# Associar as descrições aos respectivos cadastros
pares_detalhados = (
    pares_para_revisar
    .merge(originais, on="Descricao original", how="left")
    .merge(candidatos, on="Descricao candidata", how="left")
)

# Identificar se os componentes são compatíveis
pares_detalhados["Mesmo fornecedor"] = (
    pares_detalhados["Fornecedor original"]
    == pares_detalhados["Fornecedor candidato"]
)

pares_detalhados["Mesmo grupo"] = (
    pares_detalhados["Grupo original"]
    == pares_detalhados["Grupo candidato"]
)

pares_detalhados["Mesma unidade"] = (
    pares_detalhados["Unidade original"]
    == pares_detalhados["Unidade candidata"]
)

print("Total de linhas após associar os cadastros:",
      len(pares_detalhados))

print("\nCompatibilidade dos componentes:")
display(
    pares_detalhados[
        ["Mesmo fornecedor", "Mesmo grupo", "Mesma unidade"]
    ].value_counts().to_frame("Quantidade")
)

print("\nExemplos com mesmo fornecedor, grupo e unidade:")
display(
    pares_detalhados[
        pares_detalhados["Mesmo fornecedor"]
        & pares_detalhados["Mesmo grupo"]
        & pares_detalhados["Mesma unidade"]
    ][
        [
            "Código original",
            "Descricao original",
            "Código candidato",
            "Descricao candidata",
            "Similaridade",
            "Fornecedor original",
            "Grupo original",
            "Unidade original"
        ]
    ].head(20)
)

Total de linhas após associar os cadastros: 9033

Compatibilidade dos componentes:


Quantidade
Mesmo fornecedor Mesmo grupo Mesma unidade            
True             True        True                 5550
False            True        True                 3317
                 False       True                   94
True             True        False                  44
False            True        False                  23
                 False       False                   5


Exemplos com mesmo fornecedor, grupo e unidade:


,Código original,Descricao original,Código candidato,Descricao candidata,Similaridade,Fornecedor original,Grupo original,Unidade original
0,4119085,"Telha galvinizada TP40 com espess.0,5mm",4119086,"Telha galvinizada TP40 com espess.0,65mm",98.73,201813190,Civil,SV
8,4028346,SV - ANCORAGEM QUÍMICA SISTEMA HVA DA H,4028341,SV - ANCORAGEM QUÍMICA SISTEMA HVA DA HI,98.73,200303818,Civil,CM
9,4028346,SV - ANCORAGEM QUÍMICA SISTEMA HVA DA H,4028342,SV - ANCORAGEM QUÍMICA SISTEMA HVA DA HI,98.73,200303818,Civil,CM
10,4028346,SV - ANCORAGEM QUÍMICA SISTEMA HVA DA H,4028343,SV - ANCORAGEM QUÍMICA SISTEMA HVA DA HI,98.73,200303818,Civil,CM
11,4028346,SV - ANCORAGEM QUÍMICA SISTEMA HVA DA H,4028344,SV - ANCORAGEM QUÍMICA SISTEMA HVA DA HI,98.73,200303818,Civil,CM
12,4028346,SV - ANCORAGEM QUÍMICA SISTEMA HVA DA H,4028345,SV - ANCORAGEM QUÍMICA SISTEMA HVA DA HI,98.73,200303818,Civil,CM
13,4028346,SV - ANCORAGEM QUÍMICA SISTEMA HVA DA H,4028347,SV - ANCORAGEM QUÍMICA SISTEMA HVA DA HI,98.73,200303818,Civil,CM
14,4028348,SV - ANCORAGEM QUÍMICA SISTEMA HVA DA H,4028334,SV - ANCORAGEM QUÍMICA SISTEMA HVA DA HI,98.73,200303818,Civil,SV
15,4028348,SV - ANCORAGEM QUÍMICA SISTEMA HVA DA H,4028335,SV - ANCORAGEM QUÍMICA SISTEMA HVA DA HI,98.73,200303818,Civil,SV
16,4028348,SV - ANCORAGEM QUÍMICA SISTEMA HVA DA H,4028336,SV - ANCORAGEM QUÍMICA SISTEMA HVA DA HI,98.73,200303818,Civil,SV


In [28]:

# Filtrar pares em que fornecedor, grupo e unidade coincidem

pares_mesmo_contexto = pares_detalhados[
    pares_detalhados["Mesmo fornecedor"]
    & pares_detalhados["Mesmo grupo"]
    & pares_detalhados["Mesma unidade"]
].copy()

# Manter uma linha por par de descrições e contexto
chaves = [
    "Descricao original",
    "Descricao candidata",
    "Fornecedor original",
    "Grupo original",
    "Unidade original"
]

pares_mesmo_contexto = pares_mesmo_contexto.drop_duplicates(
    subset=chaves
)

print(
    "Pares distintos no mesmo contexto:",
    len(pares_mesmo_contexto)
)

print(
    "Fornecedores envolvidos:",
    pares_mesmo_contexto["Fornecedor original"].nunique()
)

# Ordenar para revisar primeiro os textos mais semelhantes
pares_mesmo_contexto = pares_mesmo_contexto.sort_values(
    "Similaridade",
    ascending=False
)

display(
    pares_mesmo_contexto[
        [
            "Fornecedor original",
            "Grupo original",
            "Unidade original",
            "Código original",
            "Descricao original",
            "Código candidato",
            "Descricao candidata",
            "Similaridade"
        ]
    ].head(30)
)

Pares distintos no mesmo contexto: 4902
Fornecedores envolvidos: 65


,Fornecedor original,Grupo original,Unidade original,Código original,Descricao original,Código candidato,Descricao candidata,Similaridade
0,201813190,Civil,SV,4119085,"Telha galvinizada TP40 com espess.0,5mm",4119086,"Telha galvinizada TP40 com espess.0,65mm",98.73
8,200303818,Civil,CM,4028346,SV - ANCORAGEM QUÍMICA SISTEMA HVA DA H,4028341,SV - ANCORAGEM QUÍMICA SISTEMA HVA DA HI,98.73
14,200303818,Civil,SV,4028348,SV - ANCORAGEM QUÍMICA SISTEMA HVA DA H,4028334,SV - ANCORAGEM QUÍMICA SISTEMA HVA DA HI,98.73
27,201398744,Facilities,H,4036816,SV-LIMP E CONSERV. Aux de Serviç HE 100%,4036814,SV-LIMP E CONSERV.Aux de Serviç HE 100%,98.73
31,200303818,Civil,CM,4028328,SV - ANCORAGEM QUÍMICA SISTEMA HIT - FOR,4028327,SV - ANCORAGEM QUÍMICA SISTEMA HIT- FOR,98.73
37,200346880,Facilities,M2,4030419,"""SV - FORRO PVC MODULAR 625 x 625MM OU 6",4030445,SV - FORRO PVC MODULAR 625 x 625MM OU 6,98.73
38,200775708,Civil,M2,4058056,SV-PINTURA TINTA ESMALTE BASE ÁGUA - ES,4058072,SV-PINTURA TINTA ESMALTE BASE ÁGUA - ESQ,98.73
39,200303818,Civil,CM,4028312,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,4028310,SV - FURAÇÃO EM CONCRETO COM COROA DIAMA,98.73
65,201954780,Facilities,SV,4110196,Inst. e Ativ. de cam. IP Dome - 4 cam.,4110199,Inst. e Ativ. de cam. IP Dome - 64 cam.,98.70
66,201954780,Facilities,SV,4110194,Inst. e Ativ. de cam. IP Dome - 1 cam.,4110197,Inst. e Ativ. de cam. IP Dome - 16 cam.,98.70


In [36]:

# Recomeçar a análise diretamente da base original

import pandas as pd

# Agrupar descrições originais pela versão normalizada
resumo_normalizacao = (
    df.groupby("Descricao_Normalizada", dropna=False)
    .agg(
        Quantidade_descricoes=("Descrição do Item", "nunique"),
        Descricoes_originais=(
            "Descrição do Item",
            lambda coluna: list(pd.unique(coluna))
        ),
        Quantidade_registros=("Descrição do Item", "size")
    )
    .reset_index()
)

# Manter apenas grupos que possuem duas ou mais descrições originais
grupos_variacao = resumo_normalizacao[
    resumo_normalizacao["Quantidade_descricoes"] > 1
].copy()

grupos_variacao = grupos_variacao.sort_values(
    "Quantidade_descricoes",
    ascending=False
)

print("Grupos com variações de escrita:", len(grupos_variacao))

print(
    "Registros envolvidos nesses grupos:",
    grupos_variacao["Quantidade_registros"].sum()
)

display(grupos_variacao.head(20))

Grupos com variações de escrita: 37
Registros envolvidos nesses grupos: 108


,Descricao_Normalizada,Quantidade_descricoes,Descricoes_originais,Quantidade_registros
211,armador - hora normal a disposicao,2,"[ARMADOR - HORA NORMAL A DISPOSICAO, ARMADOR -...",2
230,artifice,2,"[ARTIFICE, ARTÍFICE]",4
700,carpinteiro - hora normal a disposicao,2,"[CARPINTEIRO - HORA NORMAL A DISPOSICAO, CARPI...",3
755,combustivel,2,"[COMBUSTÍVEL, COMBUSTIVEL]",2
1575,h.e.df-aux. de sv. gerais -adm,2,"[H.E.DF-Aux. de Sv. Gerais -ADM, H.E.DF-Aux. ...",2
1593,h.e.df-of. de manut. predial nr35-adm,2,"[H.E.DF-Of. de Manut. Predial NR35-ADM, H.E.DF...",2
1610,h.e.ss-aux. de sv. gerais -adm,2,"[H.E.SS-Aux. de Sv. Gerais -ADM, H.E.SS-Aux. ...",2
1628,h.e.ss-of. de manut. predial nr35-adm,2,"[H.E.SS-Of. de Manut. Predial NR35-ADM, H.E.S...",2
1683,h.i+a.not-aux. de sv. gerais -adm,2,"[H.I+A.Not-Aux. de Sv. Gerais -ADM, H.I+A.Not...",2
1719,h.i+e.df-aux. de sv. gerais -adm,2,"[H.I+E.DF-Aux. de Sv. Gerais -ADM, H.I+E.DF-A...",2


In [37]:

# Relacionar as variações de escrita aos cadastros originais

colunas = [
    "Descricao_Normalizada",
    "Grupo de Serviço",
    "Código do Fornecedor",
    "Código do Serviço",
    "Descrição do Item",
    "Unidade medida"
]

registros_variacao = df.merge(
    grupos_variacao[["Descricao_Normalizada"]],
    on="Descricao_Normalizada",
    how="inner"
)[colunas].drop_duplicates()

# Resumo por descrição normalizada
resumo_codigos = (
    registros_variacao
    .groupby("Descricao_Normalizada")
    .agg(
        Quantidade_descricoes=("Descrição do Item", "nunique"),
        Quantidade_codigos=("Código do Serviço", "nunique"),
        Quantidade_fornecedores=("Código do Fornecedor", "nunique"),
        Quantidade_unidades=("Unidade medida", "nunique"),
        Codigos=("Código do Serviço", lambda s: sorted(s.unique().tolist()))
    )
    .reset_index()
)

# Priorizar grupos com mais de um código de serviço
resumo_codigos = resumo_codigos.sort_values(
    ["Quantidade_codigos", "Quantidade_descricoes"],
    ascending=False
)

print(
    "Grupos com mais de um código de serviço:",
    (resumo_codigos["Quantidade_codigos"] > 1).sum()
)

print(
    "Grupos com mais de um fornecedor:",
    (resumo_codigos["Quantidade_fornecedores"] > 1).sum()
)

display(resumo_codigos.head(20))

print("\nRegistros correspondentes aos primeiros grupos:")

grupos_prioritarios = resumo_codigos.head(10)[
    "Descricao_Normalizada"
]

display(
    registros_variacao[
        registros_variacao["Descricao_Normalizada"].isin(
            grupos_prioritarios
        )
    ].sort_values(
        ["Descricao_Normalizada", "Código do Fornecedor", "Código do Serviço"]
    )
)

Grupos com mais de um código de serviço: 37
Grupos com mais de um fornecedor: 8


,Descricao_Normalizada,Quantidade_descricoes,Quantidade_codigos,Quantidade_fornecedores,Quantidade_unidades,Codigos
1,artifice,2,4,4,1,"[4092270, 4099911, 4100964, 4117541]"
27,peric.mes - aux. de sv. gerais - adm,2,4,1,1,"[4085954, 4085991, 4086228, 4086285]"
34,tecnico de seguranca (hora normal),2,4,10,1,"[4076416, 4083623, 4089078, 4089824]"
35,tecnico de seguranca (hora not),2,4,9,1,"[4076417, 4083624, 4089079, 4089825]"
2,carpinteiro - hora normal a disposicao,2,3,3,1,"[4086449, 4096445, 4100183]"
26,mes-of. de manut. predial nr35-adm,2,3,1,1,"[4085737, 4085742, 4085755]"
28,peric.mes - aux. de sv. gerais - xy,2,3,1,1,"[4085753, 4086051, 4086322]"
31,sv-adaptador soldavel com flange p/cx. d,2,3,1,1,"[4023817, 4023818, 4023821]"
32,sv-demolicao de concreto armado com mart,2,3,1,1,"[4023880, 4023881, 4023882]"
33,tecnico de seguranca,2,3,4,1,"[4068215, 4077899, 4090816]"



Registros correspondentes aos primeiros grupos:


,Descricao_Normalizada,Grupo de Serviço,Código do Fornecedor,Código do Serviço,Descrição do Item,Unidade medida
59,artifice,Facilities,200364916,4099911,ARTIFICE,SV
60,artifice,Facilities,200733106,4100964,ARTÍFICE,SV
102,artifice,Facilities,200880612,4092270,ARTÍFICE,SV
103,artifice,Facilities,201418204,4117541,ARTIFICE,SV
8,carpinteiro - hora normal a disposicao,Civil,200772986,4086449,CARPINTEIRO - HORA NORMAL A DISPOSICAO,H
10,carpinteiro - hora normal a disposicao,Civil,201332350,4096445,CARPINTEIRO - HORA NORMAL À DISPOSIÇÃO,H
12,carpinteiro - hora normal a disposicao,Civil,300343622,4100183,CARPINTEIRO - HORA NORMAL À DISPOSIÇÃO,H
42,mes-of. de manut. predial nr35-adm,Facilities,300380700,4085737,Mês-Of. de Manut. Predial NR35-ADM,SV
43,mes-of. de manut. predial nr35-adm,Facilities,300380700,4085742,Mês-Of. de Manut. Predial NR35-ADM,SV
46,mes-of. de manut. predial nr35-adm,Facilities,300380700,4085755,Mês-Of. de Manut. Predial NR35-ADM,SV


In [38]:

# Analisar descrições que possuem mais de um código
# dentro do mesmo contexto de cadastro.

chaves_contexto = [
    "Descricao_Normalizada",
    "Grupo de Serviço",
    "Código do Fornecedor",
    "Unidade medida"
]

analise_contexto = (
    df.groupby(chaves_contexto, dropna=False)
    .agg(
        Quantidade_codigos=("Código do Serviço", "nunique"),
        Codigos=("Código do Serviço", lambda s: sorted(s.unique().tolist())),
        Descricoes_originais=(
            "Descrição do Item",
            lambda s: list(pd.unique(s))
        ),
        Quantidade_registros=("Código do Serviço", "size")
    )
    .reset_index()
)

possiveis_duplicidades = analise_contexto[
    analise_contexto["Quantidade_codigos"] > 1
].copy()

possiveis_duplicidades = possiveis_duplicidades.sort_values(
    ["Quantidade_codigos", "Quantidade_registros"],
    ascending=False
)

print(
    "Grupos com mais de um código no mesmo contexto:",
    len(possiveis_duplicidades)
)

display(possiveis_duplicidades.head(30))

Grupos com mais de um código no mesmo contexto: 341


,Descricao_Normalizada,Grupo de Serviço,Código do Fornecedor,Unidade medida,Quantidade_codigos,Codigos,Descricoes_originais,Quantidade_registros
5004,sv - furacao em concreto com coroa diam,Civil,200303818,CM,13,"[4028312, 4028313, 4028314, 4028316, 4028317, ...",[SV - FURAÇÃO EM CONCRETO COM COROA DIAM],13
4884,sv - ancoragem quimica sistema hva da hi,Civil,200303818,SV,7,"[4028334, 4028335, 4028336, 4028337, 4028338, ...",[SV - ANCORAGEM QUÍMICA SISTEMA HVA DA HI],7
4879,sv - ancoragem quimica sistema hit - for,Civil,200303818,CM,6,"[4028328, 4028329, 4028330, 4028331, 4028332, ...",[SV - ANCORAGEM QUÍMICA SISTEMA HIT - FOR],6
4883,sv - ancoragem quimica sistema hva da hi,Civil,200303818,CM,6,"[4028341, 4028342, 4028343, 4028344, 4028345, ...",[SV - ANCORAGEM QUÍMICA SISTEMA HVA DA HI],6
3227,mes-servente de obras -adm,Facilities,300380700,SV,5,"[4085736, 4085743, 4085745, 4085746, 4085769]",[Mês-Servente de obras -ADM],5
8418,sv-porta prancheta completa para pintura,Civil,300056198,ST,5,"[4023938, 4023939, 4023941, 4023942, 4023943]",[SV-PORTA PRANCHETA COMPLETA PARA PINTURA],5
2040,h.e.df-servente de obras -adm,Facilities,300380700,SV,4,"[4085916, 4085923, 4085925, 4085949]",[H.E.DF-Servente de obras -ADM],4
2075,h.e.ss-servente de obras -adm,Facilities,300380700,SV,4,"[4085855, 4085856, 4085863, 4085894]",[H.E.SS-Servente de obras -ADM],4
2149,h.i+a.not-servente de obras -adm,Facilities,300380700,SV,4,"[4086268, 4086269, 4086276, 4086300]",[H.I+A.Not-Servente de obras -ADM],4
2184,h.i+e.df-servente de obras -adm,Facilities,300380700,SV,4,"[4086210, 4086211, 4086218, 4086244]",[H.I+E.DF-Servente de obras -ADM],4


In [39]:
# Verificar os casos em que existe mais de um código
# dentro do mesmo contexto e observar os registros individualmente.

casos_prioritarios = df.merge(
    possiveis_duplicidades[
        [
            "Descricao_Normalizada",
            "Grupo de Serviço",
            "Código do Fornecedor",
            "Unidade medida"
        ]
    ],
    on=[
        "Descricao_Normalizada",
        "Grupo de Serviço",
        "Código do Fornecedor",
        "Unidade medida"
    ],
    how="inner"
)

casos_prioritarios = casos_prioritarios[
    [
        "Grupo de Serviço",
        "Código do Fornecedor",
        "Código do Serviço",
        "Descrição do Item",
        "Descricao_Normalizada",
        "Unidade medida"
    ]
].drop_duplicates()

casos_prioritarios = casos_prioritarios.sort_values(
    [
        "Grupo de Serviço",
        "Código do Fornecedor",
        "Descricao_Normalizada",
        "Código do Serviço"
    ]
)

print("Total de registros nos casos prioritários:", len(casos_prioritarios))

display(casos_prioritarios.head(50))

Total de registros nos casos prioritários: 775


,Grupo de Serviço,Código do Fornecedor,Código do Serviço,Descrição do Item,Descricao_Normalizada,Unidade medida
144,Civil,200303818,4060407,FORNECIMENTO E APLICAÇÃO DE MASTIQUE EM,fornecimento e aplicacao de mastique em,CDM
145,Civil,200303818,4060420,FORNECIMENTO E APLICAÇÃO DE MASTIQUE EM,fornecimento e aplicacao de mastique em,CDM
23,Civil,200303818,4028328,SV - ANCORAGEM QUÍMICA SISTEMA HIT - FOR,sv - ancoragem quimica sistema hit - for,CM
24,Civil,200303818,4028329,SV - ANCORAGEM QUÍMICA SISTEMA HIT - FOR,sv - ancoragem quimica sistema hit - for,CM
25,Civil,200303818,4028330,SV - ANCORAGEM QUÍMICA SISTEMA HIT - FOR,sv - ancoragem quimica sistema hit - for,CM
26,Civil,200303818,4028331,SV - ANCORAGEM QUÍMICA SISTEMA HIT - FOR,sv - ancoragem quimica sistema hit - for,CM
27,Civil,200303818,4028332,SV - ANCORAGEM QUÍMICA SISTEMA HIT - FOR,sv - ancoragem quimica sistema hit - for,CM
28,Civil,200303818,4028333,SV - ANCORAGEM QUÍMICA SISTEMA HIT - FOR,sv - ancoragem quimica sistema hit - for,CM
29,Civil,200303818,4028334,SV - ANCORAGEM QUÍMICA SISTEMA HVA DA HI,sv - ancoragem quimica sistema hva da hi,SV
31,Civil,200303818,4028335,SV - ANCORAGEM QUÍMICA SISTEMA HVA DA HI,sv - ancoragem quimica sistema hva da hi,SV


In [42]:

# Conferir os valores reais e reconstruir a distribuição.

print("Menor quantidade de códigos:",
      possiveis_duplicidades["Quantidade_codigos"].min())

print("Maior quantidade de códigos:",
      possiveis_duplicidades["Quantidade_codigos"].max())

print("\nDistribuição correta:")

distribuicao_codigos = (
    possiveis_duplicidades["Quantidade_codigos"]
    .value_counts()
    .sort_index()
    .rename_axis("Quantidade de códigos")
    .reset_index(name="Quantidade de grupos")
)

display(distribuicao_codigos)

print(
    "\nTotal de grupos:",
    distribuicao_codigos["Quantidade de grupos"].sum()
)

Menor quantidade de códigos: 2
Maior quantidade de códigos: 13

Distribuição correta:


,Quantidade de códigos,Quantidade de grupos
0,2,293
1,3,21
2,4,21
3,5,2
4,6,2
5,7,1
6,13,1



Total de grupos: 341


In [43]:

# Verificar códigos de serviço presentes em múltiplos registros.

resumo_codigos_base = (
    df.groupby("Código do Serviço")
    .agg(
        Quantidade_registros=("Código do Serviço", "size"),
        Quantidade_descricoes=("Descrição do Item", "nunique"),
        Quantidade_fornecedores=("Código do Fornecedor", "nunique"),
        Quantidade_grupos=("Grupo de Serviço", "nunique"),
        Descricoes=("Descrição do Item",
                    lambda s: list(pd.unique(s)))
    )
    .reset_index()
)

codigos_repetidos = resumo_codigos_base[
    resumo_codigos_base["Quantidade_registros"] > 1
].copy()

codigos_repetidos = codigos_repetidos.sort_values(
    "Quantidade_registros",
    ascending=False
)

print("Total de códigos de serviço distintos:", 
      df["Código do Serviço"].nunique())

print("Códigos presentes em múltiplos registros:",
      len(codigos_repetidos))

print("Códigos associados a mais de uma descrição:",
      (codigos_repetidos["Quantidade_descricoes"] > 1).sum())

display(codigos_repetidos.head(20))

Total de códigos de serviço distintos: 7937
Códigos presentes em múltiplos registros: 1234
Códigos associados a mais de uma descrição: 1


,Código do Serviço,Quantidade_registros,Quantidade_descricoes,Quantidade_fornecedores,Quantidade_grupos,Descricoes
4020,4083562,6,1,6,1,[MECÂNICO DE MANUT. (HORA EXTRA DOM/FER)]
4017,4083559,6,1,6,1,[MECÂNICO DE MANUTENÇÃO (HORA NORMAL)]
4019,4083561,6,1,6,1,[MECÂNICO DE MANUTENÇÃO (HORA EXTRA SEM)]
4083,4083625,6,1,6,1,[TÉCNICO DE SEGURANÇA (HORA EXTRA SEM)]
4068,4083610,6,1,6,1,[ELETRICISTA DE MANUT. (HE DOM/FER)]
4065,4083607,6,1,6,1,[ELETRICISTA DE MANUTENÇÃO (HORA NORMAL)]
4067,4083609,6,1,6,1,[ELETRICISTA DE MANUT. (HORA EXTRA SEM)]
6073,4097684,6,1,6,1,[PREMIO ASSIDUIDADE]
4041,4083583,6,1,6,1,[SOLDADOR (HORA NORMAL)]
4084,4083626,6,1,6,1,[TÉCNICO DE SEGURANÇA (HE DOM/FER)]


In [44]:

# Identifica os códigos associados a mais de uma descrição

casos_codigo_descricao = resumo_codigos_base[
    resumo_codigos_base["Quantidade_descricoes"] > 1
].copy()

display(
    casos_codigo_descricao[
        [
            "Código do Serviço",
            "Quantidade_registros",
            "Quantidade_descricoes",
            "Quantidade_fornecedores",
            "Quantidade_grupos",
            "Descricoes"
        ]
    ]
)

# Exibe todos os registros da base relacionados a esses códigos
codigos_investigar = casos_codigo_descricao["Código do Serviço"].tolist()

registros_codigo_investigar = df[
    df["Código do Serviço"].isin(codigos_investigar)
].sort_values("Código do Serviço")

display(registros_codigo_investigar)

,Código do Serviço,Quantidade_registros,Quantidade_descricoes,Quantidade_fornecedores,Quantidade_grupos,Descricoes
1321,4048071,3,2,3,1,"[CONTROLADOR DE ACESSO 24HS, VIGILANTE DIURNO ..."


,Grupo de Serviço,Código do Fornecedor,Código do Serviço,Descrição do Item,Unidade medida,Descricao_Normalizada
7157,Facilities,200323568,4048071,CONTROLADOR DE ACESSO 24HS,SV,controlador de acesso 24hs
7158,Facilities,200344608,4048071,VIGILANTE DIURNO 12X36,SV,vigilante diurno 12x36
7159,Facilities,200954432,4048071,VIGILANTE DIURNO 12X36,SV,vigilante diurno 12x36


In [45]:

# Investiga o código 4048071 e suas duas descrições

codigo_investigado = 4048071

registros_codigo = df[
    df["Código do Serviço"] == codigo_investigado
].copy()

print("REGISTROS DO CÓDIGO INVESTIGADO")
display(
    registros_codigo[
        [
            "Grupo de Serviço",
            "Código do Fornecedor",
            "Código do Serviço",
            "Descrição do Item",
            "Unidade medida"
        ]
    ]
)

# Procura todos os registros com as descrições encontradas
descricoes_investigadas = (
    registros_codigo["Descrição do Item"].dropna().unique()
)

print("OUTROS CÓDIGOS ASSOCIADOS ÀS MESMAS DESCRIÇÕES")

registros_descricoes = df[
    df["Descrição do Item"].isin(descricoes_investigadas)
].copy()

resumo_descricoes = (
    registros_descricoes
    .groupby("Descrição do Item")
    .agg(
        Quantidade_registros=("Código do Serviço", "size"),
        Quantidade_codigos=("Código do Serviço", "nunique"),
        Quantidade_fornecedores=("Código do Fornecedor", "nunique"),
        Codigos=("Código do Serviço",
                 lambda s: sorted(s.unique().tolist()))
    )
    .reset_index()
)

display(resumo_descricoes)

REGISTROS DO CÓDIGO INVESTIGADO


,Grupo de Serviço,Código do Fornecedor,Código do Serviço,Descrição do Item,Unidade medida
7157,Facilities,200323568,4048071,CONTROLADOR DE ACESSO 24HS,SV
7158,Facilities,200344608,4048071,VIGILANTE DIURNO 12X36,SV
7159,Facilities,200954432,4048071,VIGILANTE DIURNO 12X36,SV


OUTROS CÓDIGOS ASSOCIADOS ÀS MESMAS DESCRIÇÕES


,Descrição do Item,Quantidade_registros,Quantidade_codigos,Quantidade_fornecedores,Codigos
0,CONTROLADOR DE ACESSO 24HS,1,1,1,[4048071]
1,VIGILANTE DIURNO 12X36,2,1,2,[4048071]


In [46]:

# Identifica códigos associados a mais de uma descrição
codigos_com_descricoes_diferentes = resumo_codigos_base[
    resumo_codigos_base["Quantidade_descricoes"] > 1
].copy()

print(
    "Quantidade de códigos associados a mais de uma descrição:",
    len(codigos_com_descricoes_diferentes)
)

# Exibe os casos encontrados, do maior para o menor número de registros
display(
    codigos_com_descricoes_diferentes[
        [
            "Código do Serviço",
            "Quantidade_registros",
            "Quantidade_descricoes",
            "Quantidade_fornecedores",
            "Quantidade_grupos",
            "Descricoes"
        ]
    ].sort_values(
        ["Quantidade_descricoes", "Quantidade_registros"],
        ascending=False
    )
)

Quantidade de códigos associados a mais de uma descrição: 1


,Código do Serviço,Quantidade_registros,Quantidade_descricoes,Quantidade_fornecedores,Quantidade_grupos,Descricoes
1321,4048071,3,2,3,1,"[CONTROLADOR DE ACESSO 24HS, VIGILANTE DIURNO ..."


In [47]:

# Resume os grupos com uma mesma descrição normalizada
# associados a vários códigos no mesmo contexto.

print("Total de grupos com múltiplos códigos:", len(possiveis_duplicidades))

print("\nDistribuição pela quantidade de códigos:")

display(
    possiveis_duplicidades["Quantidade_codigos"]
    .value_counts()
    .sort_index()
    .rename_axis("Quantidade de códigos")
    .reset_index(name="Quantidade de grupos")
)

print("\nCasos com 3 ou mais códigos:")

display(
    possiveis_duplicidades[
        possiveis_duplicidades["Quantidade_codigos"] >= 3
    ].sort_values(
        "Quantidade_codigos",
        ascending=False
    )
)

Total de grupos com múltiplos códigos: 341

Distribuição pela quantidade de códigos:


,Quantidade de códigos,Quantidade de grupos
0,2,293
1,3,21
2,4,21
3,5,2
4,6,2
5,7,1
6,13,1



Casos com 3 ou mais códigos:


,Descricao_Normalizada,Grupo de Serviço,Código do Fornecedor,Unidade medida,Quantidade_codigos,Codigos,Descricoes_originais,Quantidade_registros
5004,sv - furacao em concreto com coroa diam,Civil,200303818,CM,13,"[4028312, 4028313, 4028314, 4028316, 4028317, ...",[SV - FURAÇÃO EM CONCRETO COM COROA DIAM],13
4884,sv - ancoragem quimica sistema hva da hi,Civil,200303818,SV,7,"[4028334, 4028335, 4028336, 4028337, 4028338, ...",[SV - ANCORAGEM QUÍMICA SISTEMA HVA DA HI],7
4879,sv - ancoragem quimica sistema hit - for,Civil,200303818,CM,6,"[4028328, 4028329, 4028330, 4028331, 4028332, ...",[SV - ANCORAGEM QUÍMICA SISTEMA HIT - FOR],6
4883,sv - ancoragem quimica sistema hva da hi,Civil,200303818,CM,6,"[4028341, 4028342, 4028343, 4028344, 4028345, ...",[SV - ANCORAGEM QUÍMICA SISTEMA HVA DA HI],6
3227,mes-servente de obras -adm,Facilities,300380700,SV,5,"[4085736, 4085743, 4085745, 4085746, 4085769]",[Mês-Servente de obras -ADM],5
8418,sv-porta prancheta completa para pintura,Civil,300056198,ST,5,"[4023938, 4023939, 4023941, 4023942, 4023943]",[SV-PORTA PRANCHETA COMPLETA PARA PINTURA],5
2040,h.e.df-servente de obras -adm,Facilities,300380700,SV,4,"[4085916, 4085923, 4085925, 4085949]",[H.E.DF-Servente de obras -ADM],4
2075,h.e.ss-servente de obras -adm,Facilities,300380700,SV,4,"[4085855, 4085856, 4085863, 4085894]",[H.E.SS-Servente de obras -ADM],4
2149,h.i+a.not-servente de obras -adm,Facilities,300380700,SV,4,"[4086268, 4086269, 4086276, 4086300]",[H.I+A.Not-Servente de obras -ADM],4
2184,h.i+e.df-servente de obras -adm,Facilities,300380700,SV,4,"[4086210, 4086211, 4086218, 4086244]",[H.I+E.DF-Servente de obras -ADM],4


In [48]:

# Investiga os códigos associados à descrição de furação em concreto

caso_furacao = possiveis_duplicidades[
    possiveis_duplicidades["Descricao_Normalizada"]
    == "sv - furacao em concreto com coroa diam"
].copy()

display(caso_furacao)

# Recupera os registros originais desses códigos
codigos_furacao = caso_furacao["Codigos"].iloc[0]

registros_furacao = df[
    df["Código do Serviço"].isin(codigos_furacao)
].copy()

display(
    registros_furacao[
        [
            "Grupo de Serviço",
            "Código do Fornecedor",
            "Código do Serviço",
            "Descrição do Item",
            "Unidade medida"
        ]
    ].sort_values("Código do Serviço")
)

,Descricao_Normalizada,Grupo de Serviço,Código do Fornecedor,Unidade medida,Quantidade_codigos,Codigos,Descricoes_originais,Quantidade_registros
5004,sv - furacao em concreto com coroa diam,Civil,200303818,CM,13,"[4028312, 4028313, 4028314, 4028316, 4028317, ...",[SV - FURAÇÃO EM CONCRETO COM COROA DIAM],13


,Grupo de Serviço,Código do Fornecedor,Código do Serviço,Descrição do Item,Unidade medida
97,Civil,200303818,4028312,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,CM
100,Civil,200303818,4028313,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,CM
102,Civil,200303818,4028314,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,CM
115,Civil,200303818,4028316,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,CM
117,Civil,200303818,4028317,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,CM
125,Civil,200303818,4028318,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,CM
138,Civil,200303818,4028319,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,CM
142,Civil,200303818,4028320,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,CM
146,Civil,200303818,4028321,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,CM
150,Civil,200303818,4028322,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,CM


In [49]:

# Resume os grupos com múltiplos códigos no mesmo contexto
# e cria uma lista de casos para revisão.

relatorio_prioridades = (
    possiveis_duplicidades
    .sort_values(
        ["Quantidade_codigos", "Quantidade_registros"],
        ascending=False
    )
    .copy()
)

relatorio_prioridades["Prioridade inicial"] = (
    relatorio_prioridades["Quantidade_codigos"].apply(
        lambda quantidade:
            "Alta" if quantidade >= 5
            else "Média" if quantidade >= 3
            else "Baixa"
    )
)

colunas_relatorio = [
    "Descricao_Normalizada",
    "Grupo de Serviço",
    "Código do Fornecedor",
    "Unidade medida",
    "Quantidade_codigos",
    "Codigos",
    "Prioridade inicial"
]

display(relatorio_prioridades[colunas_relatorio].head(20))

print("\nResumo das prioridades:")
display(
    relatorio_prioridades["Prioridade inicial"]
    .value_counts()
    .rename_axis("Prioridade inicial")
    .reset_index(name="Quantidade de grupos")
)

,Descricao_Normalizada,Grupo de Serviço,Código do Fornecedor,Unidade medida,Quantidade_codigos,Codigos,Prioridade inicial
5004,sv - furacao em concreto com coroa diam,Civil,200303818,CM,13,"[4028312, 4028313, 4028314, 4028316, 4028317, ...",Alta
4884,sv - ancoragem quimica sistema hva da hi,Civil,200303818,SV,7,"[4028334, 4028335, 4028336, 4028337, 4028338, ...",Alta
4879,sv - ancoragem quimica sistema hit - for,Civil,200303818,CM,6,"[4028328, 4028329, 4028330, 4028331, 4028332, ...",Alta
4883,sv - ancoragem quimica sistema hva da hi,Civil,200303818,CM,6,"[4028341, 4028342, 4028343, 4028344, 4028345, ...",Alta
3227,mes-servente de obras -adm,Facilities,300380700,SV,5,"[4085736, 4085743, 4085745, 4085746, 4085769]",Alta
8418,sv-porta prancheta completa para pintura,Civil,300056198,ST,5,"[4023938, 4023939, 4023941, 4023942, 4023943]",Alta
2040,h.e.df-servente de obras -adm,Facilities,300380700,SV,4,"[4085916, 4085923, 4085925, 4085949]",Média
2075,h.e.ss-servente de obras -adm,Facilities,300380700,SV,4,"[4085855, 4085856, 4085863, 4085894]",Média
2149,h.i+a.not-servente de obras -adm,Facilities,300380700,SV,4,"[4086268, 4086269, 4086276, 4086300]",Média
2184,h.i+e.df-servente de obras -adm,Facilities,300380700,SV,4,"[4086210, 4086211, 4086218, 4086244]",Média



Resumo das prioridades:


,Prioridade inicial,Quantidade de grupos
0,Baixa,293
1,Média,42
2,Alta,6


In [50]:

# Seleciona os grupos de alta prioridade
grupos_alta_prioridade = relatorio_prioridades[
    relatorio_prioridades["Prioridade inicial"] == "Alta"
].copy()

# Recupera os registros originais de cada grupo
chaves = [
    "Descricao_Normalizada",
    "Grupo de Serviço",
    "Código do Fornecedor",
    "Unidade medida"
]

registros_alta_prioridade = df.merge(
    grupos_alta_prioridade[chaves],
    on=chaves,
    how="inner"
)

# Exibe os registros completos para comparação
display(
    registros_alta_prioridade[
        [
            "Grupo de Serviço",
            "Código do Fornecedor",
            "Código do Serviço",
            "Descrição do Item",
            "Unidade medida",
            "Descricao_Normalizada"
        ]
    ].sort_values(
        [
            "Descrição do Item",
            "Código do Serviço"
        ]
    )
)

,Grupo de Serviço,Código do Fornecedor,Código do Serviço,Descrição do Item,Unidade medida,Descricao_Normalizada
37,Facilities,300380700,4085736,Mês-Servente de obras -ADM,SV,mes-servente de obras -adm
38,Facilities,300380700,4085743,Mês-Servente de obras -ADM,SV,mes-servente de obras -adm
39,Facilities,300380700,4085745,Mês-Servente de obras -ADM,SV,mes-servente de obras -adm
40,Facilities,300380700,4085746,Mês-Servente de obras -ADM,SV,mes-servente de obras -adm
41,Facilities,300380700,4085769,Mês-Servente de obras -ADM,SV,mes-servente de obras -adm
12,Civil,200303818,4028328,SV - ANCORAGEM QUÍMICA SISTEMA HIT - FOR,CM,sv - ancoragem quimica sistema hit - for
13,Civil,200303818,4028329,SV - ANCORAGEM QUÍMICA SISTEMA HIT - FOR,CM,sv - ancoragem quimica sistema hit - for
14,Civil,200303818,4028330,SV - ANCORAGEM QUÍMICA SISTEMA HIT - FOR,CM,sv - ancoragem quimica sistema hit - for
15,Civil,200303818,4028331,SV - ANCORAGEM QUÍMICA SISTEMA HIT - FOR,CM,sv - ancoragem quimica sistema hit - for
16,Civil,200303818,4028332,SV - ANCORAGEM QUÍMICA SISTEMA HIT - FOR,CM,sv - ancoragem quimica sistema hit - for


In [51]:

# Relatório de possíveis códigos redundantes

relatorio_redundancias = possiveis_duplicidades.copy()

# Quantidade de códigos diferentes encontrados no grupo
relatorio_redundancias["Quantidade_codigos"] = (
    relatorio_redundancias["Quantidade_codigos"].astype(int)
)

# Classificação inicial para organizar a investigação
def classificar_prioridade(quantidade):
    if quantidade >= 5:
        return "Alta"
    elif quantidade >= 3:
        return "Média"
    return "Baixa"

relatorio_redundancias["Prioridade"] = (
    relatorio_redundancias["Quantidade_codigos"]
    .apply(classificar_prioridade)
)

# A validação precisa ser feita antes de concluir que existe erro
relatorio_redundancias["Status_validacao"] = "Pendente"

# Organiza os casos mais relevantes primeiro
relatorio_redundancias = relatorio_redundancias.sort_values(
    ["Quantidade_codigos", "Quantidade_registros"],
    ascending=False
).reset_index(drop=True)

# Visualiza as colunas mais úteis para o projeto
display(
    relatorio_redundancias[
        [
            "Descricao_Normalizada",
            "Grupo de Serviço",
            "Código do Fornecedor",
            "Unidade medida",
            "Quantidade_codigos",
            "Codigos",
            "Quantidade_registros",
            "Prioridade",
            "Status_validacao"
        ]
    ]
)

,Descricao_Normalizada,Grupo de Serviço,Código do Fornecedor,Unidade medida,Quantidade_codigos,Codigos,Quantidade_registros,Prioridade,Status_validacao
0,sv - furacao em concreto com coroa diam,Civil,200303818,CM,13,"[4028312, 4028313, 4028314, 4028316, 4028317, ...",13,Alta,Pendente
1,sv - ancoragem quimica sistema hva da hi,Civil,200303818,SV,7,"[4028334, 4028335, 4028336, 4028337, 4028338, ...",7,Alta,Pendente
2,sv - ancoragem quimica sistema hit - for,Civil,200303818,CM,6,"[4028328, 4028329, 4028330, 4028331, 4028332, ...",6,Alta,Pendente
3,sv - ancoragem quimica sistema hva da hi,Civil,200303818,CM,6,"[4028341, 4028342, 4028343, 4028344, 4028345, ...",6,Alta,Pendente
4,mes-servente de obras -adm,Facilities,300380700,SV,5,"[4085736, 4085743, 4085745, 4085746, 4085769]",5,Alta,Pendente
...,...,...,...,...,...,...,...,...,...
336,tecnico seguranca(hora extra dom e fer),Eletromecânica,300745236,H,2,"[4076419, 4089083]",2,Baixa,Pendente
337,tecnico seguranca(hora extra not dom.fer,Eletromecânica,300745236,H,2,"[4076689, 4089081]",2,Baixa,Pendente
338,tecnico seguranca(hora extra not seg.sab,Eletromecânica,300745236,H,2,"[4076688, 4089080]",2,Baixa,Pendente
339,tecnico seguranca(hora extra seg a sab),Eletromecânica,300745236,H,2,"[4076418, 4089082]",2,Baixa,Pendente


In [53]:

from rapidfuzz import process, fuzz
import pandas as pd

def buscar_servicos(descricao, limite=10, similaridade_minima=60):
    """
    Busca descrições semelhantes na base histórica.
    O percentual indica semelhança textual, não equivalência comercial.
    """

    descricao_normalizada = normalizar_descricao(descricao)

    resultados = process.extract(
        descricao_normalizada,
        df["Descricao_Normalizada"].drop_duplicates().tolist(),
        scorer=fuzz.ratio,
        limit=limite,
        score_cutoff=similaridade_minima
    )

    linhas = []

    for descricao_encontrada, pontuacao, _ in resultados:
        registros = df[
            df["Descricao_Normalizada"] == descricao_encontrada
        ]

        for _, registro in registros.iterrows():
            linhas.append({
                "Similaridade (%)": round(pontuacao, 1),
                "Grupo de Serviço": registro["Grupo de Serviço"],
                "Código do Fornecedor": registro["Código do Fornecedor"],
                "Código do Serviço": registro["Código do Serviço"],
                "Descrição do Item": registro["Descrição do Item"],
                "Unidade medida": registro["Unidade medida"]
            })

    if not linhas:
        return pd.DataFrame(
            columns=[
                "Similaridade (%)",
                "Grupo de Serviço",
                "Código do Fornecedor",
                "Código do Serviço",
                "Descrição do Item",
                "Unidade medida"
            ]
        )

    return (
        pd.DataFrame(linhas)
        .drop_duplicates()
        .sort_values("Similaridade (%)", ascending=False)
        .reset_index(drop=True)
    )

In [54]:

resultado_busca = buscar_servicos(
    "SV - FURAÇÃO EM CONCRETO COM COROA DIAM",
    limite=10,
    similaridade_minima=60
)

display(resultado_busca)

,Similaridade (%),Grupo de Serviço,Código do Fornecedor,Código do Serviço,Descrição do Item,Unidade medida
0,100.0,Civil,200303818,4028312,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,CM
1,100.0,Civil,200303818,4028313,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,CM
2,100.0,Civil,200303818,4028314,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,CM
3,100.0,Civil,200303818,4028316,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,CM
4,100.0,Civil,200303818,4028317,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,CM
5,100.0,Civil,200303818,4028318,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,CM
6,100.0,Civil,200303818,4028319,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,CM
7,100.0,Civil,200303818,4028320,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,CM
8,100.0,Civil,200303818,4028321,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,CM
9,100.0,Civil,200303818,4028322,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,CM


In [55]:

# Agrupa os resultados por descrição e contexto do serviço

resumo_busca = (
    resultado_busca
    .groupby(
        [
            "Similaridade (%)",
            "Grupo de Serviço",
            "Código do Fornecedor",
            "Descrição do Item",
            "Unidade medida"
        ],
        dropna=False
    )
    .agg(
        Codigos_encontrados=(
            "Código do Serviço",
            lambda s: sorted(s.unique().tolist())
        ),
        Quantidade_codigos=(
            "Código do Serviço",
            "nunique"
        )
    )
    .reset_index()
    .sort_values(
        ["Similaridade (%)", "Quantidade_codigos"],
        ascending=False
    )
)

display(resumo_busca)

,Similaridade (%),Grupo de Serviço,Código do Fornecedor,Descrição do Item,Unidade medida,Codigos_encontrados,Quantidade_codigos
10,100.0,Civil,200303818,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,CM,"[4028312, 4028313, 4028314, 4028316, 4028317, ...",13
9,98.7,Civil,200303818,SV - FURAÇÃO EM CONCRETO COM COROA DIAMA,CM,"[4028310, 4028315]",2
8,93.7,Civil,200303818,SV-FURAÇÃO EM CONCRETO COM COROA DIAMANT,CM,[4028311],1
7,83.5,Civil,300056198,SV-FURAÇÃO EM CONCRETO COM BROCA DE VIDE,CM,[4023908],1
4,71.1,Civil,200303818,"FURAÇÃO EM BLOCO CARBONO COM COROA 2""",CM,[4060383],1
5,71.1,Civil,200303818,"FURAÇÃO EM BLOCO CARBONO COM COROA 3""",CM,[4060384],1
6,71.1,Civil,200303818,"FURAÇÃO EM BLOCO CARBONO COM COROA 4""",CM,[4060385],1
2,69.2,Civil,200303818,"FURAÇÃO EM BLOCO CARBONO COM COROA 1/2""",CM,[4060321],1
3,69.2,Civil,200303818,"FURAÇÃO EM BLOCO CARBONO COM COROA 3/4""",CM,[4060320],1
0,62.2,Civil,200490010,SV - TUBO DE CONCRETO ARMADO 600 mm,M,[4050227],1


In [81]:

from rapidfuzz import process, fuzz
import pandas as pd

COLUNAS_RESULTADO = [
    "Similaridade (%)",
    "Grupo de Serviço",
    "Código do Fornecedor",
    "Código do Serviço",
    "Descrição do Item",
    "Unidade medida"
]

def buscar_servicos_contextual(
    descricao,
    grupo=None,
    fornecedor=None,
    unidade=None,
    limite=10,
    similaridade_minima=60
):
    descricao_normalizada = normalizar_descricao(descricao)
    candidatos = df.copy()

    # Aplica os filtros informados
    if grupo is not None:
        candidatos = candidatos[
            candidatos["Grupo de Serviço"].astype(str).str.casefold()
            == str(grupo).casefold()
        ]

    if fornecedor is not None:
        candidatos = candidatos[
            candidatos["Código do Fornecedor"].astype(str)
            == str(fornecedor)
        ]

    if unidade is not None:
        candidatos = candidatos[
            candidatos["Unidade medida"].astype(str).str.strip().str.casefold()
            == str(unidade).strip().casefold()
        ]

    # Retorna uma tabela vazia com as colunas esperadas
    # caso nenhum registro passe pelos filtros
    if candidatos.empty:
        return pd.DataFrame(columns=COLUNAS_RESULTADO)

    descricoes = (
        candidatos["Descricao_Normalizada"]
        .drop_duplicates()
        .tolist()
    )

    correspondencias = process.extract(
        descricao_normalizada,
        descricoes,
        scorer=fuzz.ratio,
        limit=limite,
        score_cutoff=similaridade_minima
    )

    # Trata consultas sem correspondências suficientes
    if not correspondencias:
        return pd.DataFrame(columns=COLUNAS_RESULTADO)

    resultados = []

    for descricao_encontrada, pontuacao, _ in correspondencias:
        registros = candidatos[
            candidatos["Descricao_Normalizada"] == descricao_encontrada
        ]

        for _, registro in registros.iterrows():
            resultados.append({
                "Similaridade (%)": round(pontuacao, 1),
                "Grupo de Serviço": registro["Grupo de Serviço"],
                "Código do Fornecedor": registro["Código do Fornecedor"],
                "Código do Serviço": registro["Código do Serviço"],
                "Descrição do Item": registro["Descrição do Item"],
                "Unidade medida": registro["Unidade medida"]
            })

    if not resultados:
        return pd.DataFrame(columns=COLUNAS_RESULTADO)

    return (
        pd.DataFrame(resultados, columns=COLUNAS_RESULTADO)
        .drop_duplicates()
        .sort_values("Similaridade (%)", ascending=False)
        .head(50)
        .reset_index(drop=True)
    )

In [58]:

resultado_contextual = buscar_servicos_contextual(
    descricao="SV - FURAÇÃO EM CONCRETO COM COROA DIAM",
    grupo="Civil",
    fornecedor=200303818,
    unidade="CM",
    limite=10,
    similaridade_minima=60
)

display(resultado_contextual)

,Similaridade (%),Grupo de Serviço,Código do Fornecedor,Código do Serviço,Descrição do Item,Unidade medida
0,100.0,Civil,200303818,4028312,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,CM
1,100.0,Civil,200303818,4028313,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,CM
2,100.0,Civil,200303818,4028314,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,CM
3,100.0,Civil,200303818,4028316,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,CM
4,100.0,Civil,200303818,4028317,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,CM
5,100.0,Civil,200303818,4028318,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,CM
6,100.0,Civil,200303818,4028319,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,CM
7,100.0,Civil,200303818,4028320,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,CM
8,100.0,Civil,200303818,4028321,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,CM
9,100.0,Civil,200303818,4028322,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,CM


In [59]:

import re

def extrair_numeros(texto):
    """
    Extrai números inteiros, decimais e frações
    presentes em uma descrição.
    """
    texto = str(texto).lower().replace(",", ".")

    padrao = r"\d+(?:\.\d+)?(?:/\d+(?:\.\d+)?)?"

    return re.findall(padrao, texto)


# Teste com descrições de serviços diferentes
exemplos = [
    "FURAÇÃO EM BLOCO CARBONO COM COROA 2 polegadas",
    "FURAÇÃO EM BLOCO CARBONO COM COROA 3 polegadas",
    "TELHA METÁLICA 0,5 mm",
    "TELHA METÁLICA 0,65 mm",
    "SERVIÇO DE FURAÇÃO EM CONCRETO"
]

for descricao in exemplos:
    print(f"Descrição: {descricao}")
    print(f"Números encontrados: {extrair_numeros(descricao)}")
    print("-" * 50)

Descrição: FURAÇÃO EM BLOCO CARBONO COM COROA 2 polegadas
Números encontrados: ['2']
--------------------------------------------------
Descrição: FURAÇÃO EM BLOCO CARBONO COM COROA 3 polegadas
Números encontrados: ['3']
--------------------------------------------------
Descrição: TELHA METÁLICA 0,5 mm
Números encontrados: ['0.5']
--------------------------------------------------
Descrição: TELHA METÁLICA 0,65 mm
Números encontrados: ['0.65']
--------------------------------------------------
Descrição: SERVIÇO DE FURAÇÃO EM CONCRETO
Números encontrados: []
--------------------------------------------------


In [60]:

def comparar_numeros(descricao_a, descricao_b):
    numeros_a = extrair_numeros(descricao_a)
    numeros_b = extrair_numeros(descricao_b)

    if numeros_a == numeros_b:
        return "Números iguais"

    return "Revisar: números diferentes"


# Testes
testes = [
    (
        "FURAÇÃO EM BLOCO CARBONO COM COROA 2 polegadas",
        "FURAÇÃO EM BLOCO CARBONO COM COROA 3 polegadas"
    ),
    (
        "TELHA METÁLICA 0,5 mm",
        "TELHA METÁLICA 0,65 mm"
    ),
    (
        "SV - FURAÇÃO EM CONCRETO COM COROA DIAM",
        "SV - FURAÇÃO EM CONCRETO COM COROA DIAMA"
    ),
    (
        "SERVIÇO DE FURAÇÃO EM CONCRETO",
        "SERVIÇO DE FURAÇÃO EM CONCRETO"
    )
]

for descricao_a, descricao_b in testes:
    print(f"A: {descricao_a}")
    print(f"B: {descricao_b}")
    print(f"Resultado: {comparar_numeros(descricao_a, descricao_b)}")
    print("-" * 70)

A: FURAÇÃO EM BLOCO CARBONO COM COROA 2 polegadas
B: FURAÇÃO EM BLOCO CARBONO COM COROA 3 polegadas
Resultado: Revisar: números diferentes
----------------------------------------------------------------------
A: TELHA METÁLICA 0,5 mm
B: TELHA METÁLICA 0,65 mm
Resultado: Revisar: números diferentes
----------------------------------------------------------------------
A: SV - FURAÇÃO EM CONCRETO COM COROA DIAM
B: SV - FURAÇÃO EM CONCRETO COM COROA DIAMA
Resultado: Números iguais
----------------------------------------------------------------------
A: SERVIÇO DE FURAÇÃO EM CONCRETO
B: SERVIÇO DE FURAÇÃO EM CONCRETO
Resultado: Números iguais
----------------------------------------------------------------------


In [61]:

def buscar_servicos_classificados(
    descricao,
    grupo=None,
    fornecedor=None,
    unidade=None,
    limite=10,
    similaridade_minima=60
):
    resultados = buscar_servicos_contextual(
        descricao=descricao,
        grupo=grupo,
        fornecedor=fornecedor,
        unidade=unidade,
        limite=limite,
        similaridade_minima=similaridade_minima
    )

    if resultados.empty:
        return resultados

    descricao_normalizada = normalizar_descricao(descricao)
    numeros_pesquisados = extrair_numeros(descricao)

    def classificar_linha(linha):
        descricao_candidata = linha["Descrição do Item"]
        numeros_candidatos = extrair_numeros(descricao_candidata)

        if normalizar_descricao(descricao_candidata) == descricao_normalizada:
            if numeros_pesquisados == numeros_candidatos:
                return "Correspondência textual exata"

        if numeros_pesquisados != numeros_candidatos:
            return "Revisar diferenças numéricas"

        return "Verificar contexto do serviço"

    resultados["Classificação inicial"] = resultados.apply(
        classificar_linha,
        axis=1
    )

    return resultados

In [62]:

resultado_classificado = buscar_servicos_classificados(
    descricao="SV - FURAÇÃO EM CONCRETO COM COROA DIAM",
    grupo="Civil",
    fornecedor=200303818,
    unidade="CM",
    limite=10,
    similaridade_minima=60
)

display(resultado_classificado)

,Similaridade (%),Grupo de Serviço,Código do Fornecedor,Código do Serviço,Descrição do Item,Unidade medida,Classificação inicial
0,100.0,Civil,200303818,4028312,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,CM,Correspondência textual exata
1,100.0,Civil,200303818,4028313,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,CM,Correspondência textual exata
2,100.0,Civil,200303818,4028314,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,CM,Correspondência textual exata
3,100.0,Civil,200303818,4028316,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,CM,Correspondência textual exata
4,100.0,Civil,200303818,4028317,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,CM,Correspondência textual exata
5,100.0,Civil,200303818,4028318,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,CM,Correspondência textual exata
6,100.0,Civil,200303818,4028319,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,CM,Correspondência textual exata
7,100.0,Civil,200303818,4028320,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,CM,Correspondência textual exata
8,100.0,Civil,200303818,4028321,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,CM,Correspondência textual exata
9,100.0,Civil,200303818,4028322,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,CM,Correspondência textual exata


In [63]:

# Casos de teste conhecidos para avaliar a busca

casos_teste = [
    {
        "descricao": "SV - FURAÇÃO EM CONCRETO COM COROA DIAM",
        "grupo": "Civil",
        "fornecedor": 200303818,
        "unidade": "CM",
        "codigo_esperado": 4028312,
        "objetivo": "Encontrar um código existente com descrição exata"
    },
    {
        "descricao": "SV - FURAÇÃO EM CONCRETO COM COROA DIAMA",
        "grupo": "Civil",
        "fornecedor": 200303818,
        "unidade": "CM",
        "codigo_esperado": 4028310,
        "objetivo": "Encontrar uma variação de descrição existente"
    },
    {
        "descricao": "FURAÇÃO EM BLOCO CARBONO COM COROA 2 polegadas",
        "grupo": "Civil",
        "fornecedor": 200303818,
        "unidade": "CM",
        "codigo_esperado": 4060383,
        "objetivo": "Encontrar um serviço com medida específica"
    }
]

resultados_teste = []

for caso in casos_teste:
    encontrados = buscar_servicos_contextual(
        descricao=caso["descricao"],
        grupo=caso["grupo"],
        fornecedor=caso["fornecedor"],
        unidade=caso["unidade"],
        limite=10,
        similaridade_minima=60
    )

    codigos = (
        encontrados["Código do Serviço"].astype(int).tolist()
        if not encontrados.empty
        else []
    )

    resultados_teste.append({
        "Objetivo": caso["objetivo"],
        "Código esperado": caso["codigo_esperado"],
        "Código encontrado": caso["codigo_esperado"] in codigos,
        "Quantidade de resultados": len(encontrados)
    })

avaliacao_busca = pd.DataFrame(resultados_teste)

display(avaliacao_busca)

,Objetivo,Código esperado,Código encontrado,Quantidade de resultados
0,Encontrar um código existente com descrição exata,4028312,True,21
1,Encontrar uma variação de descrição existente,4028310,True,21
2,Encontrar um serviço com medida específica,4060383,True,21


In [64]:

for caso in casos_teste:
    encontrados = buscar_servicos_contextual(
        descricao=caso["descricao"],
        grupo=caso["grupo"],
        fornecedor=caso["fornecedor"],
        unidade=caso["unidade"],
        limite=10,
        similaridade_minima=60
    )

    print("\n" + "=" * 70)
    print("Objetivo:", caso["objetivo"])
    print("Código esperado:", caso["codigo_esperado"])

    if encontrados.empty:
        print("Nenhum resultado encontrado.")
        continue

    exibicao = encontrados[
        [
            "Similaridade (%)",
            "Código do Serviço",
            "Descrição do Item",
            "Unidade medida"
        ]
    ].copy()

    exibicao["Posição"] = range(1, len(exibicao) + 1)

    posicao = exibicao.index[
        exibicao["Código do Serviço"].astype(int)
        == caso["codigo_esperado"]
    ]

    if len(posicao) > 0:
        print("Posição do código esperado:", exibicao.loc[posicao[0], "Posição"])
    else:
        print("Código esperado não encontrado na exibição.")

    display(exibicao.head(10))


Objetivo: Encontrar um código existente com descrição exata
Código esperado: 4028312
Posição do código esperado: 1


,Similaridade (%),Código do Serviço,Descrição do Item,Unidade medida,Posição
0,100.0,4028312,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,CM,1
1,100.0,4028313,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,CM,2
2,100.0,4028314,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,CM,3
3,100.0,4028316,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,CM,4
4,100.0,4028317,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,CM,5
5,100.0,4028318,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,CM,6
6,100.0,4028319,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,CM,7
7,100.0,4028320,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,CM,8
8,100.0,4028321,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,CM,9
9,100.0,4028322,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,CM,10



Objetivo: Encontrar uma variação de descrição existente
Código esperado: 4028310
Posição do código esperado: 1


,Similaridade (%),Código do Serviço,Descrição do Item,Unidade medida,Posição
0,100.0,4028310,SV - FURAÇÃO EM CONCRETO COM COROA DIAMA,CM,1
1,100.0,4028315,SV - FURAÇÃO EM CONCRETO COM COROA DIAMA,CM,2
2,98.7,4028312,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,CM,3
3,98.7,4028313,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,CM,4
4,98.7,4028314,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,CM,5
5,98.7,4028316,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,CM,6
6,98.7,4028317,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,CM,7
7,98.7,4028318,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,CM,8
8,98.7,4028319,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,CM,9
9,98.7,4028320,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,CM,10



Objetivo: Encontrar um serviço com medida específica
Código esperado: 4060383
Posição do código esperado: 1


,Similaridade (%),Código do Serviço,Descrição do Item,Unidade medida,Posição
0,86.7,4060383,"FURAÇÃO EM BLOCO CARBONO COM COROA 2""",CM,1
1,84.7,4060321,"FURAÇÃO EM BLOCO CARBONO COM COROA 1/2""",CM,2
2,84.3,4060384,"FURAÇÃO EM BLOCO CARBONO COM COROA 3""",CM,3
3,84.3,4060385,"FURAÇÃO EM BLOCO CARBONO COM COROA 4""",CM,4
4,82.4,4060320,"FURAÇÃO EM BLOCO CARBONO COM COROA 3/4""",CM,5
5,68.2,4028312,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,CM,6
6,68.2,4028313,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,CM,7
7,68.2,4028314,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,CM,8
8,68.2,4028316,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,CM,9
9,68.2,4028317,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,CM,10


In [65]:

def buscar_servicos_com_analise(
    descricao,
    grupo=None,
    fornecedor=None,
    unidade=None,
    limite=10,
    similaridade_minima=60
):
    resultados = buscar_servicos_contextual(
        descricao=descricao,
        grupo=grupo,
        fornecedor=fornecedor,
        unidade=unidade,
        limite=limite,
        similaridade_minima=similaridade_minima
    )

    if resultados.empty:
        return resultados

    numeros_pesquisados = extrair_numeros(descricao)
    descricao_normalizada = normalizar_descricao(descricao)

    def analisar_resultado(linha):
        descricao_candidata = linha["Descrição do Item"]
        numeros_candidatos = extrair_numeros(descricao_candidata)

        if normalizar_descricao(descricao_candidata) == descricao_normalizada:
            return "Descrição exata"

        if numeros_pesquisados != numeros_candidatos:
            return "Atenção: números diferentes"

        return "Revisar similaridade e contexto"

    resultados["Análise inicial"] = resultados.apply(
        analisar_resultado, axis=1
    )

    return resultados


teste_medidas = buscar_servicos_com_analise(
    descricao="FURAÇÃO EM BLOCO CARBONO COM COROA 2 polegadas",
    grupo="Civil",
    fornecedor=200303818,
    unidade="CM",
    limite=10,
    similaridade_minima=60
)

display(
    teste_medidas[
        [
            "Similaridade (%)",
            "Código do Serviço",
            "Descrição do Item",
            "Análise inicial"
        ]
    ].head(10)
)

,Similaridade (%),Código do Serviço,Descrição do Item,Análise inicial
0,86.7,4060383,"FURAÇÃO EM BLOCO CARBONO COM COROA 2""",Revisar similaridade e contexto
1,84.7,4060321,"FURAÇÃO EM BLOCO CARBONO COM COROA 1/2""",Atenção: números diferentes
2,84.3,4060384,"FURAÇÃO EM BLOCO CARBONO COM COROA 3""",Atenção: números diferentes
3,84.3,4060385,"FURAÇÃO EM BLOCO CARBONO COM COROA 4""",Atenção: números diferentes
4,82.4,4060320,"FURAÇÃO EM BLOCO CARBONO COM COROA 3/4""",Atenção: números diferentes
5,68.2,4028312,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,Atenção: números diferentes
6,68.2,4028313,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,Atenção: números diferentes
7,68.2,4028314,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,Atenção: números diferentes
8,68.2,4028316,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,Atenção: números diferentes
9,68.2,4028317,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,Atenção: números diferentes


In [66]:

import re

def extrair_numeros(texto):
    texto = str(texto).lower().replace(",", ".")

    # Padroniza formatos de polegadas.
    texto = re.sub(r"\bpolegadas?\b", " pol", texto)
    texto = texto.replace('"', " pol")

    # Extrai números inteiros, decimais e frações.
    padrao = r"\d+(?:\.\d+)?(?:/\d+(?:\.\d+)?)?"
    return re.findall(padrao, texto)


# Teste da nova extração
testes_medidas = [
    'FURAÇÃO EM BLOCO CARBONO COM COROA 2 polegadas',
    'FURAÇÃO EM BLOCO CARBONO COM COROA 2"',
    'FURAÇÃO EM BLOCO CARBONO COM COROA 1/2"',
    'FURAÇÃO EM BLOCO CARBONO COM COROA 3/4"'
]

for descricao in testes_medidas:
    print(descricao, "->", extrair_numeros(descricao))

FURAÇÃO EM BLOCO CARBONO COM COROA 2 polegadas -> ['2']
FURAÇÃO EM BLOCO CARBONO COM COROA 2" -> ['2']
FURAÇÃO EM BLOCO CARBONO COM COROA 1/2" -> ['1/2']
FURAÇÃO EM BLOCO CARBONO COM COROA 3/4" -> ['3/4']


In [67]:

def buscar_servicos_com_analise(
    descricao,
    grupo=None,
    fornecedor=None,
    unidade=None,
    limite=10,
    similaridade_minima=60
):
    resultados = buscar_servicos_contextual(
        descricao=descricao,
        grupo=grupo,
        fornecedor=fornecedor,
        unidade=unidade,
        limite=limite,
        similaridade_minima=similaridade_minima
    )

    if resultados.empty:
        return resultados

    descricao_normalizada = normalizar_descricao(descricao)
    numeros_pesquisados = extrair_numeros(descricao)

    def comparar_texto(descricao_candidata):
        if normalizar_descricao(descricao_candidata) == descricao_normalizada:
            return "Descrição normalizada exata"
        return "Descrição semelhante"

    def comparar_medidas(descricao_candidata):
        numeros_candidatos = extrair_numeros(descricao_candidata)

        if numeros_pesquisados == numeros_candidatos:
            return "Números correspondentes"

        return "Atenção: números diferentes"

    resultados["Comparação textual"] = resultados[
        "Descrição do Item"
    ].apply(comparar_texto)

    resultados["Comparação numérica"] = resultados[
        "Descrição do Item"
    ].apply(comparar_medidas)

    return resultados

In [68]:

teste_medidas = buscar_servicos_com_analise(
    descricao="FURAÇÃO EM BLOCO CARBONO COM COROA 2 polegadas",
    grupo="Civil",
    fornecedor=200303818,
    unidade="CM",
    limite=10,
    similaridade_minima=60
)

display(
    teste_medidas[
        [
            "Similaridade (%)",
            "Código do Serviço",
            "Descrição do Item",
            "Comparação textual",
            "Comparação numérica"
        ]
    ].head(10)
)

,Similaridade (%),Código do Serviço,Descrição do Item,Comparação textual,Comparação numérica
0,86.7,4060383,"FURAÇÃO EM BLOCO CARBONO COM COROA 2""",Descrição semelhante,Números correspondentes
1,84.7,4060321,"FURAÇÃO EM BLOCO CARBONO COM COROA 1/2""",Descrição semelhante,Atenção: números diferentes
2,84.3,4060384,"FURAÇÃO EM BLOCO CARBONO COM COROA 3""",Descrição semelhante,Atenção: números diferentes
3,84.3,4060385,"FURAÇÃO EM BLOCO CARBONO COM COROA 4""",Descrição semelhante,Atenção: números diferentes
4,82.4,4060320,"FURAÇÃO EM BLOCO CARBONO COM COROA 3/4""",Descrição semelhante,Atenção: números diferentes
5,68.2,4028312,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,Descrição semelhante,Atenção: números diferentes
6,68.2,4028313,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,Descrição semelhante,Atenção: números diferentes
7,68.2,4028314,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,Descrição semelhante,Atenção: números diferentes
8,68.2,4028316,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,Descrição semelhante,Atenção: números diferentes
9,68.2,4028317,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,Descrição semelhante,Atenção: números diferentes


In [69]:

def buscar_servicos_com_analise(
    descricao,
    grupo=None,
    fornecedor=None,
    unidade=None,
    limite=10,
    similaridade_minima=60
):
    resultados = buscar_servicos_contextual(
        descricao=descricao,
        grupo=grupo,
        fornecedor=fornecedor,
        unidade=unidade,
        limite=limite,
        similaridade_minima=similaridade_minima
    )

    if resultados.empty:
        return resultados

    descricao_normalizada = normalizar_descricao(descricao)
    numeros_pesquisados = extrair_numeros(descricao)

    def comparar_texto(descricao_candidata):
        if normalizar_descricao(descricao_candidata) == descricao_normalizada:
            return "Descrição normalizada exata"
        return "Descrição semelhante"

    def comparar_numeros(descricao_candidata):
        numeros_candidatos = extrair_numeros(descricao_candidata)

        if numeros_pesquisados == numeros_candidatos:
            return "Números correspondentes"
        return "Números diferentes"

    resultados["Comparação textual"] = resultados[
        "Descrição do Item"
    ].apply(comparar_texto)

    resultados["Comparação numérica"] = resultados[
        "Descrição do Item"
    ].apply(comparar_numeros)

    def classificar(linha):
        if (
            linha["Comparação textual"] == "Descrição normalizada exata"
            and linha["Comparação numérica"] == "Números correspondentes"
        ):
            return "Correspondência textual exata"

        if linha["Comparação numérica"] == "Números diferentes":
            return "Revisar diferenças numéricas"

        return "Candidato para revisão"

    resultados["Classificação inicial"] = resultados.apply(
        classificar, axis=1
    )

    return resultados


teste_medidas = buscar_servicos_com_analise(
    descricao="FURAÇÃO EM BLOCO CARBONO COM COROA 2 polegadas",
    grupo="Civil",
    fornecedor=200303818,
    unidade="CM",
    limite=10,
    similaridade_minima=60
)

display(
    teste_medidas[
        [
            "Similaridade (%)",
            "Código do Serviço",
            "Descrição do Item",
            "Comparação numérica",
            "Classificação inicial"
        ]
    ].head(10)
)

,Similaridade (%),Código do Serviço,Descrição do Item,Comparação numérica,Classificação inicial
0,86.7,4060383,"FURAÇÃO EM BLOCO CARBONO COM COROA 2""",Números correspondentes,Candidato para revisão
1,84.7,4060321,"FURAÇÃO EM BLOCO CARBONO COM COROA 1/2""",Números diferentes,Revisar diferenças numéricas
2,84.3,4060384,"FURAÇÃO EM BLOCO CARBONO COM COROA 3""",Números diferentes,Revisar diferenças numéricas
3,84.3,4060385,"FURAÇÃO EM BLOCO CARBONO COM COROA 4""",Números diferentes,Revisar diferenças numéricas
4,82.4,4060320,"FURAÇÃO EM BLOCO CARBONO COM COROA 3/4""",Números diferentes,Revisar diferenças numéricas
5,68.2,4028312,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,Números diferentes,Revisar diferenças numéricas
6,68.2,4028313,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,Números diferentes,Revisar diferenças numéricas
7,68.2,4028314,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,Números diferentes,Revisar diferenças numéricas
8,68.2,4028316,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,Números diferentes,Revisar diferenças numéricas
9,68.2,4028317,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,Números diferentes,Revisar diferenças numéricas


In [70]:

casos_teste = [
    {
        "descricao": "SV - FURAÇÃO EM CONCRETO COM COROA DIAM",
        "grupo": "Civil",
        "fornecedor": 200303818,
        "unidade": "CM",
        "codigo_esperado": 4028312,
        "objetivo": "Descrição exata"
    },
    {
        "descricao": "SV - FURAÇÃO EM CONCRETO COM COROA DIAMA",
        "grupo": "Civil",
        "fornecedor": 200303818,
        "unidade": "CM",
        "codigo_esperado": 4028310,
        "objetivo": "Variação textual"
    },
    {
        "descricao": "FURAÇÃO EM BLOCO CARBONO COM COROA 2 polegadas",
        "grupo": "Civil",
        "fornecedor": 200303818,
        "unidade": "CM",
        "codigo_esperado": 4060383,
        "objetivo": "Medida específica"
    }
]

avaliacoes = []

for caso in casos_teste:
    resultados = buscar_servicos_com_analise(
        descricao=caso["descricao"],
        grupo=caso["grupo"],
        fornecedor=caso["fornecedor"],
        unidade=caso["unidade"],
        limite=10,
        similaridade_minima=60
    )

    codigos = (
        resultados["Código do Serviço"].astype(int).tolist()
        if not resultados.empty
        else []
    )

    posicoes = [
        i + 1
        for i, codigo in enumerate(codigos)
        if codigo == caso["codigo_esperado"]
    ]

    avaliacoes.append({
        "Cenário": caso["objetivo"],
        "Código esperado": caso["codigo_esperado"],
        "Encontrado": bool(posicoes),
        "Melhor posição": min(posicoes) if posicoes else None,
        "Total de resultados": len(resultados)
    })

avaliacao_busca = pd.DataFrame(avaliacoes)
display(avaliacao_busca)

,Cenário,Código esperado,Encontrado,Melhor posição,Total de resultados
0,Descrição exata,4028312,True,1,21
1,Variação textual,4028310,True,1,21
2,Medida específica,4060383,True,1,21


In [72]:

# Reorganiza as colunas e garante a apresentação correta.
avaliacao_busca = pd.DataFrame(avaliacoes, columns=[
    "Cenário",
    "Código esperado",
    "Encontrado",
    "Melhor posição",
    "Total de resultados"
])

display(avaliacao_busca.to_string(index=False))

# Métricas dos testes realizados
total_testes = len(avaliacao_busca)
total_encontrados = int(avaliacao_busca["Encontrado"].sum())
taxa_encontrados = (
    total_encontrados / total_testes * 100
    if total_testes else 0
)

print(f"\nCódigos esperados encontrados: {total_encontrados}/{total_testes}")
print(f"Taxa de recuperação nestes testes: {taxa_encontrados:.1f}%")

'          Cenário  Código esperado  Encontrado  Melhor posição  Total de resultados\n  Descrição exata          4028312        True               1                   21\n Variação textual          4028310        True               1                   21\nMedida específica          4060383        True               1                   21'


Códigos esperados encontrados: 3/3
Taxa de recuperação nestes testes: 100.0%


In [74]:

avaliacao_detalhada = []

for caso in casos_teste:
    resultados = buscar_servicos_com_analise(
        descricao=caso["descricao"],
        grupo=caso["grupo"],
        fornecedor=caso["fornecedor"],
        unidade=caso["unidade"],
        limite=10,
        similaridade_minima=60
    )

    if resultados.empty:
        codigos_unicos = []
        posicao = None
    else:
        # Mantém a ordem dos códigos na lista de resultados.
        codigos_unicos = (
            resultados["Código do Serviço"]
            .drop_duplicates()
            .astype(int)
            .tolist()
        )

        posicoes = [
            i + 1
            for i, codigo in enumerate(codigos_unicos)
            if codigo == caso["codigo_esperado"]
        ]

        posicao = min(posicoes) if posicoes else None

    avaliacao_detalhada.append({
        "Cenário": caso["objetivo"],
        "Código esperado": caso["codigo_esperado"],
        "Encontrado": caso["codigo_esperado"] in codigos_unicos,
        "Posição entre códigos únicos": posicao,
        "Códigos distintos retornados": len(codigos_unicos)
    })

display(pd.DataFrame(avaliacao_detalhada).to_string(index=False))

'          Cenário  Código esperado  Encontrado  Posição entre códigos únicos  Códigos distintos retornados\n  Descrição exata          4028312        True                             1                            21\n Variação textual          4028310        True                             1                            21\nMedida específica          4060383        True                             1                            21'

In [75]:

caso = casos_teste[2]

resultados = buscar_servicos_com_analise(
    descricao=caso["descricao"],
    grupo=caso["grupo"],
    fornecedor=caso["fornecedor"],
    unidade=caso["unidade"],
    limite=10,
    similaridade_minima=60
)

# Um registro por código de serviço, mantendo a maior similaridade.
resultados_unicos = (
    resultados
    .sort_values("Similaridade (%)", ascending=False)
    .drop_duplicates(subset=["Código do Serviço"])
)

display(
    resultados_unicos[
        [
            "Similaridade (%)",
            "Código do Serviço",
            "Descrição do Item",
            "Comparação numérica",
            "Classificação inicial"
        ]
    ].head(25).to_string(index=False)
)

' Similaridade (%)  Código do Serviço                        Descrição do Item     Comparação numérica        Classificação inicial\n             86.7            4060383    FURAÇÃO EM BLOCO CARBONO COM COROA 2" Números correspondentes       Candidato para revisão\n             84.7            4060321  FURAÇÃO EM BLOCO CARBONO COM COROA 1/2"      Números diferentes Revisar diferenças numéricas\n             84.3            4060384    FURAÇÃO EM BLOCO CARBONO COM COROA 3"      Números diferentes Revisar diferenças numéricas\n             84.3            4060385    FURAÇÃO EM BLOCO CARBONO COM COROA 4"      Números diferentes Revisar diferenças numéricas\n             82.4            4060320  FURAÇÃO EM BLOCO CARBONO COM COROA 3/4"      Números diferentes Revisar diferenças numéricas\n             68.2            4028312  SV - FURAÇÃO EM CONCRETO COM COROA DIAM      Números diferentes Revisar diferenças numéricas\n             68.2            4028313  SV - FURAÇÃO EM CONCRETO COM COROA DI

In [76]:

# Resume os resultados do teste de medida específica.
total_resultados = len(resultados_unicos)

numeros_correspondentes = (
    resultados_unicos["Comparação numérica"]
    == "Números correspondentes"
).sum()

numeros_diferentes = (
    resultados_unicos["Comparação numérica"]
    == "Números diferentes"
).sum()

print("Total de códigos distintos:", total_resultados)
print("Candidatos com números correspondentes:", numeros_correspondentes)
print("Candidatos com números diferentes:", numeros_diferentes)

if total_resultados:
    print(
        "Percentual com números correspondentes:",
        f"{numeros_correspondentes / total_resultados * 100:.1f}%"
    )
    print(
        "Percentual com números diferentes:",
        f"{numeros_diferentes / total_resultados * 100:.1f}%"
    )

# Agrupa os resultados por classificação inicial.
display(
    resultados_unicos["Classificação inicial"]
    .value_counts()
    .rename_axis("Classificação")
    .reset_index(name="Quantidade")
)

Total de códigos distintos: 21
Candidatos com números correspondentes: 1
Candidatos com números diferentes: 20
Percentual com números correspondentes: 4.8%
Percentual com números diferentes: 95.2%


,Classificação,Quantidade
0,Revisar diferenças numéricas,20
1,Candidato para revisão,1


In [78]:

resultados_ordenados = resultados_unicos.copy()

def definir_prioridade(linha):
    texto_exato = (
        linha["Comparação textual"] == "Descrição normalizada exata"
    )
    numeros_iguais = (
        linha["Comparação numérica"] == "Números correspondentes"
    )

    if texto_exato and numeros_iguais:
        return 1
    elif numeros_iguais:
        return 2
    else:
        return 3

resultados_ordenados["Prioridade_busca"] = (
    resultados_ordenados.apply(definir_prioridade, axis=1)
)

resultados_ordenados = resultados_ordenados.sort_values(
    by=["Prioridade_busca", "Similaridade (%)"],
    ascending=[True, False]
).reset_index(drop=True)

display(
    resultados_ordenados[
        [
            "Código do Serviço",
            "Descrição do Item",
            "Similaridade (%)",
            "Comparação textual",
            "Comparação numérica",
            "Classificação inicial",
            "Prioridade_busca"
        ]
    ]
)

,Código do Serviço,Descrição do Item,Similaridade (%),Comparação textual,Comparação numérica,Classificação inicial,Prioridade_busca
0,4060383,"FURAÇÃO EM BLOCO CARBONO COM COROA 2""",86.7,Descrição semelhante,Números correspondentes,Candidato para revisão,2
1,4060321,"FURAÇÃO EM BLOCO CARBONO COM COROA 1/2""",84.7,Descrição semelhante,Números diferentes,Revisar diferenças numéricas,3
2,4060384,"FURAÇÃO EM BLOCO CARBONO COM COROA 3""",84.3,Descrição semelhante,Números diferentes,Revisar diferenças numéricas,3
3,4060385,"FURAÇÃO EM BLOCO CARBONO COM COROA 4""",84.3,Descrição semelhante,Números diferentes,Revisar diferenças numéricas,3
4,4060320,"FURAÇÃO EM BLOCO CARBONO COM COROA 3/4""",82.4,Descrição semelhante,Números diferentes,Revisar diferenças numéricas,3
5,4028312,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,68.2,Descrição semelhante,Números diferentes,Revisar diferenças numéricas,3
6,4028313,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,68.2,Descrição semelhante,Números diferentes,Revisar diferenças numéricas,3
7,4028314,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,68.2,Descrição semelhante,Números diferentes,Revisar diferenças numéricas,3
8,4028316,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,68.2,Descrição semelhante,Números diferentes,Revisar diferenças numéricas,3
9,4028317,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,68.2,Descrição semelhante,Números diferentes,Revisar diferenças numéricas,3


In [79]:

# Verifica se a prioridade foi calculada corretamente
print("Distribuição das prioridades:")
display(
    resultados_ordenados["Prioridade_busca"]
    .value_counts()
    .sort_index()
    .rename_axis("Prioridade")
    .reset_index(name="Quantidade")
)

print("\nCandidatos com prioridade 1 ou 2:")
display(
    resultados_ordenados[
        resultados_ordenados["Prioridade_busca"].isin([1, 2])
    ][
        [
            "Código do Serviço",
            "Descrição do Item",
            "Similaridade (%)",
            "Comparação textual",
            "Comparação numérica",
            "Prioridade_busca"
        ]
    ]
)

Distribuição das prioridades:


,Prioridade,Quantidade
0,2,1
1,3,20



Candidatos com prioridade 1 ou 2:


,Código do Serviço,Descrição do Item,Similaridade (%),Comparação textual,Comparação numérica,Prioridade_busca
0,4060383,"FURAÇÃO EM BLOCO CARBONO COM COROA 2""",86.7,Descrição semelhante,Números correspondentes,2


In [82]:

# TESTE 1: descrição já existente na base
consulta_exata = "SV - FURAÇÃO EM CONCRETO COM COROA DIAM"

# TESTE 2: descrição com pequenas diferenças de escrita
consulta_variacao = "furacao em bloco carbono com coroa 2 polegadas"

# TESTE 3: descrição provavelmente inexistente na base
consulta_inedita = "manutencao de drone subaquatico para inspecao industrial"

consultas_teste = [
    ("Descrição existente", consulta_exata),
    ("Variação de escrita", consulta_variacao),
    ("Descrição inédita", consulta_inedita)
]

for nome_teste, consulta in consultas_teste:
    print(f"\n{'=' * 70}")
    print(f"TESTE: {nome_teste}")
    print(f"Consulta: {consulta}")

    resultados_teste = buscar_servicos_com_analise(
        descricao=consulta,
        limite=10,
        similaridade_minima=60
    )

    if resultados_teste.empty:
        print("Nenhum candidato encontrado.")
    else:
        resultados_teste_unicos = (
            resultados_teste
            .sort_values("Similaridade (%)", ascending=False)
            .drop_duplicates(subset=["Código do Serviço"])
        )

        display(
            resultados_teste_unicos[
                [
                    "Código do Serviço",
                    "Descrição do Item",
                    "Similaridade (%)",
                    "Comparação numérica",
                    "Classificação inicial"
                ]
            ].head(10)
        )


TESTE: Descrição existente
Consulta: SV - FURAÇÃO EM CONCRETO COM COROA DIAM


,Código do Serviço,Descrição do Item,Similaridade (%),Comparação numérica,Classificação inicial
0,4028312,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,100.0,Números correspondentes,Correspondência textual exata
1,4028313,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,100.0,Números correspondentes,Correspondência textual exata
2,4028314,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,100.0,Números correspondentes,Correspondência textual exata
3,4028316,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,100.0,Números correspondentes,Correspondência textual exata
4,4028317,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,100.0,Números correspondentes,Correspondência textual exata
5,4028318,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,100.0,Números correspondentes,Correspondência textual exata
6,4028319,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,100.0,Números correspondentes,Correspondência textual exata
7,4028320,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,100.0,Números correspondentes,Correspondência textual exata
8,4028321,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,100.0,Números correspondentes,Correspondência textual exata
9,4028322,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,100.0,Números correspondentes,Correspondência textual exata



TESTE: Variação de escrita
Consulta: furacao em bloco carbono com coroa 2 polegadas


,Código do Serviço,Descrição do Item,Similaridade (%),Comparação numérica,Classificação inicial
0,4060383,"FURAÇÃO EM BLOCO CARBONO COM COROA 2""",86.7,Números correspondentes,Candidato para revisão
1,4060321,"FURAÇÃO EM BLOCO CARBONO COM COROA 1/2""",84.7,Números diferentes,Revisar diferenças numéricas
2,4060384,"FURAÇÃO EM BLOCO CARBONO COM COROA 3""",84.3,Números diferentes,Revisar diferenças numéricas
3,4060385,"FURAÇÃO EM BLOCO CARBONO COM COROA 4""",84.3,Números diferentes,Revisar diferenças numéricas
4,4060320,"FURAÇÃO EM BLOCO CARBONO COM COROA 3/4""",82.4,Números diferentes,Revisar diferenças numéricas
5,4028312,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,68.2,Números diferentes,Revisar diferenças numéricas
6,4028313,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,68.2,Números diferentes,Revisar diferenças numéricas
7,4028314,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,68.2,Números diferentes,Revisar diferenças numéricas
8,4028316,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,68.2,Números diferentes,Revisar diferenças numéricas
9,4028317,SV - FURAÇÃO EM CONCRETO COM COROA DIAM,68.2,Números diferentes,Revisar diferenças numéricas



TESTE: Descrição inédita
Consulta: manutencao de drone subaquatico para inspecao industrial
Nenhum candidato encontrado.


In [83]:

# Seleciona descrições existentes na base para criar casos de teste
descricoes_teste = (
    df["Descrição do Item"]
    .drop_duplicates()
    .sample(n=10, random_state=42)
    .tolist()
)

resultados_avaliacao = []

for descricao in descricoes_teste:
    resultado = buscar_servicos_com_analise(
        descricao=descricao,
        limite=10,
        similaridade_minima=60
    )

    codigos_esperados = set(
        df.loc[
            df["Descricao_Normalizada"] == normalizar_descricao(descricao),
            "Código do Serviço"
        ]
    )

    codigos_encontrados = (
        set(resultado["Código do Serviço"])
        if not resultado.empty
        else set()
    )

    resultados_avaliacao.append({
        "Consulta": descricao,
        "Códigos esperados": len(codigos_esperados),
        "Códigos encontrados": len(codigos_encontrados),
        "Encontrou código esperado": bool(
            codigos_esperados & codigos_encontrados
        )
    })

avaliacao = pd.DataFrame(resultados_avaliacao)

display(avaliacao)

print("\nResumo da avaliação:")
print("Consultas testadas:", len(avaliacao))
print(
    "Consultas com pelo menos um código esperado encontrado:",
    avaliacao["Encontrou código esperado"].sum()
)
print(
    "Taxa de recuperação:",
    f"{avaliacao['Encontrou código esperado'].mean() * 100:.1f}%"
)

,Consulta,Códigos esperados,Códigos encontrados,Encontrou código esperado
0,TÉCNICO DE PLANEJAMENTO,1,10,True
1,SV - TRANSPORTE SARDINHA - FURGAO,1,11,True
2,SV RET APARELHOS SANITARIOS,1,4,True
3,Diária Manut./Resgatista-EQ.FIXA NOTURNO,1,5,True
4,SV-TAMPA EXTREMIDADE PARA CANALETA,1,7,True
5,SV-AZULEJO PAREDE 20CMx20CM C/ ARGAMASSA,1,2,True
6,SV TST HN INS CHEIA,1,10,True
7,SV VIDRO LISO 4 MILÍMETROS,1,10,True
8,ALMOXARIFE - HORA EXTRA 50%,1,10,True
9,SOLDADOR MAÇARIQUEIRO (HE NOT DOM/FER),1,15,True



Resumo da avaliação:
Consultas testadas: 10
Consultas com pelo menos um código esperado encontrado: 10
Taxa de recuperação: 100.0%


In [84]:

# Avalia quantos códigos retornados correspondem
# aos códigos associados à descrição original.

metricas_consultas = []

for descricao in descricoes_teste:
    resultado = buscar_servicos_com_analise(
        descricao=descricao,
        limite=10,
        similaridade_minima=60
    )

    codigos_esperados = set(
        df.loc[
            df["Descricao_Normalizada"] == normalizar_descricao(descricao),
            "Código do Serviço"
        ]
    )

    codigos_encontrados = (
        set(resultado["Código do Serviço"])
        if not resultado.empty
        else set()
    )

    codigos_corretos = codigos_esperados & codigos_encontrados

    metricas_consultas.append({
        "Consulta": descricao,
        "Códigos esperados": len(codigos_esperados),
        "Códigos encontrados": len(codigos_encontrados),
        "Códigos esperados recuperados": len(codigos_corretos),
        "Candidatos adicionais": len(
            codigos_encontrados - codigos_esperados
        )
    })

metricas = pd.DataFrame(metricas_consultas)

display(metricas)

print("Total de códigos adicionais apresentados:",
      metricas["Candidatos adicionais"].sum())

print("Consultas com candidatos adicionais:",
      (metricas["Candidatos adicionais"] > 0).sum())

print("Total de consultas:",
      len(metricas))

,Consulta,Códigos esperados,Códigos encontrados,Códigos esperados recuperados,Candidatos adicionais
0,TÉCNICO DE PLANEJAMENTO,1,10,1,9
1,SV - TRANSPORTE SARDINHA - FURGAO,1,11,1,10
2,SV RET APARELHOS SANITARIOS,1,4,1,3
3,Diária Manut./Resgatista-EQ.FIXA NOTURNO,1,5,1,4
4,SV-TAMPA EXTREMIDADE PARA CANALETA,1,7,1,6
5,SV-AZULEJO PAREDE 20CMx20CM C/ ARGAMASSA,1,2,1,1
6,SV TST HN INS CHEIA,1,10,1,9
7,SV VIDRO LISO 4 MILÍMETROS,1,10,1,9
8,ALMOXARIFE - HORA EXTRA 50%,1,10,1,9
9,SOLDADOR MAÇARIQUEIRO (HE NOT DOM/FER),1,15,1,14


Total de códigos adicionais apresentados: 74
Consultas com candidatos adicionais: 10
Total de consultas: 10


In [85]:

# Compara diferentes limites de similaridade
# usando as mesmas 10 consultas de teste.

limites_teste = [60, 70, 80, 90, 95]
comparacao_limites = []

for limite in limites_teste:
    total_candidatos = 0
    total_codigos_esperados = 0
    consultas_com_codigo_esperado = 0

    for descricao in descricoes_teste:
        resultado = buscar_servicos_com_analise(
            descricao=descricao,
            limite=10,
            similaridade_minima=limite
        )

        codigos_esperados = set(
            df.loc[
                df["Descricao_Normalizada"]
                == normalizar_descricao(descricao),
                "Código do Serviço"
            ]
        )

        codigos_encontrados = (
            set(resultado["Código do Serviço"])
            if not resultado.empty
            else set()
        )

        total_candidatos += len(codigos_encontrados)
        total_codigos_esperados += len(codigos_esperados)

        if codigos_esperados & codigos_encontrados:
            consultas_com_codigo_esperado += 1

    comparacao_limites.append({
        "Similaridade mínima": limite,
        "Total de códigos retornados": total_candidatos,
        "Média de códigos por consulta": round(
            total_candidatos / len(descricoes_teste), 2
        ),
        "Consultas com código esperado": consultas_com_codigo_esperado
    })

comparacao_limites_df = pd.DataFrame(comparacao_limites)
display(comparacao_limites_df)

,Similaridade mínima,Total de códigos retornados,Média de códigos por consulta,Consultas com código esperado
0,60,84,8.4,10
1,70,68,6.8,10
2,80,46,4.6,10
3,90,18,1.8,10
4,95,14,1.4,10


In [86]:
print("Funções disponíveis:")
print(normalizar_descricao.__name__)
print(buscar_servicos.__name__)
print(buscar_servicos_contextual.__name__)
print(buscar_servicos_com_analise.__name__)
print(extrair_numeros.__name__)

Funções disponíveis:
normalizar_descricao
buscar_servicos
buscar_servicos_contextual
buscar_servicos_com_analise
extrair_numeros


In [4]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

In [10]:
import sys
from pathlib import Path

projeto = Path.cwd().parent
detector = projeto / "src" / "detector"

if str(detector) not in sys.path:
    sys.path.insert(0, str(detector))

print("Pasta do detector:")
print(detector)

print("\nArquivo:")
print(detector / "normalizacao.py")

print("\nExiste:", (detector / "normalizacao.py").exists())

Pasta do detector:
c:\Users\felip\OneDrive\Área de Trabalho\T.I\Certificados\Portfolio\Desafio_GC_Gerdau\IA\src\detector

Arquivo:
c:\Users\felip\OneDrive\Área de Trabalho\T.I\Certificados\Portfolio\Desafio_GC_Gerdau\IA\src\detector\normalizacao.py

Existe: True


In [2]:
from pathlib import Path
import sys

projeto = Path.cwd().parent
detector = projeto / "src" / "detector"

if str(detector) not in sys.path:
    sys.path.insert(0, str(detector))

from normalizacao import normalizar_texto

In [3]:
texto = "DIARIA RESGATISTA NIVEL 3 ATE 8H ."

resultado = normalizar_texto(texto)

print("Original:")
print(texto)

print("\nNormalizado:")
print(resultado)

Original:
DIARIA RESGATISTA NIVEL 3 ATE 8H .

Normalizado:
diaria resgatista nivel 3 ate 8h


In [ ]:
import importlib
import fuzzy

importlib.reload(fuzzy)

from fuzzy import calcular_similaridade, classificar_similaridade

In [14]:
texto1 = "DIARIA RESGATISTA NIVEL 3 ATE 8H ."
texto2 = "Diaria resgatista turno de ate 8 horas ."

normalizado1 = normalizar_texto(texto1)
normalizado2 = normalizar_texto(texto2)

similaridade = calcular_similaridade(
    normalizado1,
    normalizado2
)

classificacao = classificar_similaridade(similaridade)

print(f"Similaridade: {similaridade:.2f}%")
print(f"Classificação: {classificacao}")

NameError: name 'classificar_similaridade' is not defined